# Catching the Quiet Signal — Colab pipeline

Interval-stratified benchmarking of command-and-control beaconing detection.
Run this top to bottom and it produces every number, table and figure the paper needs.

---

### Nothing is downloaded to your computer, and nothing has to be dropped

Two worries, both settled:

**Storage.** Everything below runs on Colab's machine, which gives you roughly 78 GB of
disk. Your own device never holds any of it. The 8.8 GB / 21 GB figures you have seen
are for the *complete* IoT-23 archive — and you do not download the complete archive.
Both corpora publish **per-scenario** files, and this notebook fetches only the scenarios
it needs. IoT-23 capture 8-1 is a 1.4 MB connection log.

**CTU-13.** It does *not* need Zeek and it does *not* need packet captures. CTU-13
publishes labelled bidirectional Argus flows under
`detailed-bidirectional-flow-labels/*.binetflow` — connection-level records carrying
start time, duration, protocol, endpoints, state, bytes, packets and a Label column.
That is everything the interval derivation reads. `src/preprocess.py` maps the Argus
schema onto the Zeek one and dispatches on file type, so both corpora flow through one
pipeline. **The cross-corpus claim in your paper stands, and you run it here.**

### What this notebook does

| | Status |
|---|---|
| IoT-23 connection logs | ✅ downloaded per scenario, a few MB each |
| CTU-13 labelled Argus flows | ✅ downloaded per scenario, no Zeek, no pcaps |
| Interval derivation + banding | ✅ the paper's contribution, runs here |
| Random Forest / k-NN baselines | ✅ runs here |
| Suricata signature baseline | ✅ installable via `apt`, runs on the IoT-23 pcaps |

One honest caveat, and it is already written into the paper: Argus records no
application-protocol annotation, so the `service` feature is absent for CTU-13 and is
imputed as unknown. Every field the derivation depends on is present in both corpora.

---
## Step 0 — Check the environment

No GPU is needed. This pipeline works on tabular flow features; the heaviest step is
replaying packet captures through Suricata, which is disk-bound.

In [ ]:
import os, sys, shutil, platform
print('Python :', sys.version.split()[0])
print('Platform:', platform.platform())

import psutil
gb = 1024**3
vm = psutil.virtual_memory(); du = shutil.disk_usage('/content')
print(f'RAM    : {vm.total/gb:.1f} GB total, {vm.available/gb:.1f} GB free')
print(f'Disk   : {du.total/gb:.1f} GB total, {du.free/gb:.1f} GB free')

if vm.total/gb < 10:
    print('\n! Low RAM. Reduce SCENARIOS in Step 3 if you hit an out-of-memory kill.')

---
## Step 1 — Install dependencies

Colab already ships numpy, pandas, scikit-learn and matplotlib. We add
`imbalanced-learn` for SMOTE. Without it the code still runs but falls back to class
weighting, which is *not* the methodology the paper describes — so install it.

In [ ]:
!pip install -q imbalanced-learn shap

import importlib
for m in ('numpy','pandas','sklearn','matplotlib','imblearn','shap'):
    try:
        mod = importlib.import_module(m)
        print(f'{m:14} {getattr(mod, "__version__", "?")}')
    except ImportError:
        print(f'{m:14} MISSING')

---
## Step 2 — Write the pipeline

Each cell below writes one module. After running them you have the full repo layout in
`/content/quiet-signal`, which you can browse and edit in Colab's file pane, and push
to GitHub as-is.

**These are the same files as the repo** — the notebook is generated from them, so they
cannot drift apart.

In [ ]:
import os, pathlib
ROOT = '/content/quiet-signal'
for d in ('src','tests','data','results','figures','paper'):
    pathlib.Path(ROOT, d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
pathlib.Path('src/__init__.py').write_text('__version__ = "1.0.0"\n')
print('working directory:', os.getcwd())

In [ ]:
%%writefile src/config.py
"""Central configuration for the Quiet Signal pipeline.

Everything the paper reports as a design decision is fixed here, in one place,
so that the manuscript and the code cannot drift apart.
"""
from __future__ import annotations

from pathlib import Path

# ---------------------------------------------------------------- paths
ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
for _p in (DATA, RESULTS, FIGURES):
    _p.mkdir(exist_ok=True)

# ---------------------------------------------------------------- corpora
# IoT-23  : Zenodo 4743746, CC BY 4.0   (primary)
# CTU-13  : Stratosphere Laboratory     (cross-corpus validation)
CORPORA = {
    "iot23": {
        "url": "https://zenodo.org/records/4743746",
        "doi": "10.5281/zenodo.4743746",
        "licence": "CC BY 4.0",
        "native_format": "conn.log.labeled",
    },
    "ctu13": {
        "url": "https://www.stratosphereips.org/datasets-ctu13",
        "licence": "research use, citation requested",
        "native_format": "pcap + bidirectional NetFlow",
    },
}

# Label vocabulary counted as command-and-control (positive class).
# Matching is substring-based and case-insensitive, and spans BOTH corpora:
#
#   IoT-23  compound Zeek labels, e.g. "C&C-HeartBeat-FileDownload"
#   CTU-13  Argus labels, e.g. "flow=From-Botnet-V46-TCP-CC12-HTTP-Not-Encrypted"
#
# CTU-13's "From-Botnet" family covers ALL botnet traffic -- spam, DDoS, port
# scans, DNS/ICMP activity -- not only command and control; across the full
# 13-scenario set, only 103 of 4,734 From-Botnet-labeled channels carry a
# CC marker (2.2%). Botnet flows that are not CC are EXCLUDED rather than
# treated as either class: calling a DDoS flow "benign" would corrupt the
# false-alarm denominator, and counting it as C2 would conflate beaconing
# with unrelated attack traffic.
#
# "-cc" alone (case-insensitive substring) covers every CC-channel label
# observed in the raw data (CC1 through CC108), verified against a full
# scan of the 13 scenarios' Label columns; "cc1"-"cc4" are redundant with
# it and kept for readability. No non-CC From-Botnet label in the observed
# taxonomy contains "-cc".
C2_LABEL_TOKENS = ("c&c", "cc-heartbeat", "heartbeat", "command and control",
                   "-cc", "cc1", "cc2", "cc3", "cc4")
BENIGN_LABEL_TOKENS = ("benign", "normal", "legitimate")
# Explicitly excluded: anything else. Background traffic, and botnet traffic
# that is not command-and-control, are dropped -- see Section III-D.
EXCLUDE_LABEL_TOKENS = ("background",)

# ---------------------------------------------------------------- channels
# A channel is a communication relationship, not a flow. Beacon period is a
# property of the relationship, so periodicity is estimated per channel.
CHANNEL_KEY = ("id.orig_h", "id.resp_h", "id.resp_p", "proto")

# A periodicity estimate from a handful of observations is not defensible.
MIN_OBSERVATIONS = 10

# ---------------------------------------------------------------- bands
# Boundaries chosen to match the operator-facing configuration granularity of
# commodity C2 tooling, and FIXED BEFORE any detector is run.
BAND_EDGES_S = (60.0, 600.0)
BANDS = ("B1", "B2", "B3")
BAND_DESC = {
    "B1": "sub-minute",
    "B2": "minutes",
    "B3": "ten minutes or more",
}

# ---------------------------------------------------------------- features
ZEEK_NUMERIC = [
    "duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts",
]
ZEEK_CATEGORICAL = ["proto", "service", "conn_state"]
TIMING_FEATURES = [
    "dt_mean", "dt_median", "dt_std", "dt_mad", "jitter_ratio",
]
# The band label is a STRATIFIER, never a model input -- including it would
# leak the very quantity whose effect the study measures. "period" and
# "corpus" join it here for the same reason once the fine-grained and
# cross-corpus analyses (src/generalize.py) are in play.
FORBIDDEN_FEATURES = {"band", "label", "channel_id", "period", "corpus"}

# ---------------------------------------------------------------- models
RANDOM_SEEDS = (11, 23, 37, 53, 71)     # five seeds; results reported mean +/- sd
N_FOLDS = 5

RF_PARAMS = dict(
    n_estimators=300,
    criterion="gini",
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
)
RF_GRID = {"max_features": ["sqrt", "log2"]}

KNN_PARAMS = dict(metric="minkowski", p=2, weights="distance", n_jobs=-1)
KNN_GRID = {"n_neighbors": [3, 5, 7, 9, 11]}

# Gradient Boosting added as a THIRD learning baseline (additive -- Random
# Forest and k-NN results are unchanged). HistGradientBoostingClassifier is
# used because it is in sklearn core (no extra dependency) and handles the
# same skewed numeric features Ghani et al. flag in IoT-23 natively via
# histogram binning.
GB_PARAMS = dict(
    max_iter=300,
    learning_rate=0.08,
    max_depth=6,
    l2_regularization=1.0,
)
GB_GRID = {"max_leaf_nodes": [15, 31, 63]}

USE_SMOTE = True        # applied to TRAINING FOLDS ONLY (Section III-H)

# ---------------------------------------------------------------- Suricata
SURICATA_BIN = "suricata"
SURICATA_RULES = DATA / "rules" / "emerging-all.rules"   # ET Open
# An alert is attributed to a flow by five-tuple within this many seconds.
ALERT_MATCH_TOLERANCE_S = 2.0

# ---------------------------------------------------------------- evaluation
# Pre-registered breakdown criterion (Eq. 9 in the paper). These values are fixed BEFORE
# results are examined. Do not revise them afterwards -- doing so voids the
# pre-registration claim made in the paper.
TAU = 0.70      # absolute detection-rate floor
DELTA = 0.20    # maximum permitted drop relative to B1
DECISION_THRESHOLD = 0.5

DETECTORS = ("Suricata", "Random Forest", "Gradient Boosting", "k-NN")

In [ ]:
%%writefile src/preprocess.py
"""Stage B -- preprocessing: parse connection records, assign classes, build
channels.

Handles both corpora through one representation, and reads each in the form its
authors released:

  IoT-23  ``conn.log.labeled``  -- Zeek TSV                 (read_conn_log)
  CTU-13  ``*.binetflow``       -- labelled Argus NetFlow   (read_binetflow)

Neither path needs Zeek installed and neither needs packet captures. Use
``read_any`` to dispatch. ``zeek_replay`` remains available for anyone starting
from raw pcaps, but the published pipeline does not call it.
"""
from __future__ import annotations

import gzip
import logging
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from . import config as C

log = logging.getLogger(__name__)

# Zeek writes "-" for unset and "(empty)" for empty sets.
NA_TOKENS = ["-", "(empty)", ""]


# --------------------------------------------------------------------------
# Reading Zeek logs
# --------------------------------------------------------------------------
def _open(path: Path):
    return gzip.open(path, "rt") if path.suffix == ".gz" else open(path, "rt")


def read_conn_log(path: Path) -> pd.DataFrame:
    """Read a Zeek conn.log or conn.log.labeled into a DataFrame.

    Zeek's TSV format carries its schema in ``#fields``. IoT-23 appends two
    extra columns to that header line; we honour whatever the header declares
    rather than assuming a fixed column count.
    """
    path = Path(path)
    fields: list[str] | None = None
    rows: list[list[str]] = []

    with _open(path) as fh:
        for line in fh:
            if line.startswith("#"):
                if line.startswith("#fields"):
                    fields = line.rstrip("\n").split("\t")[1:]
                continue
            if not line.strip():
                continue
            rows.append(line.rstrip("\n").split("\t"))

    if fields is None:
        raise ValueError(f"{path}: no #fields header; is this a Zeek log?")

    # IoT-23's label columns are sometimes space-separated in the data rows
    # even though the header is tab-separated. Normalise row width.
    width = len(fields)
    fixed = []
    for r in rows:
        if len(r) > width:
            r = r[: width - 1] + [" ".join(r[width - 1:])]
        elif len(r) < width:
            r = r + ["-"] * (width - len(r))
        fixed.append(r)

    df = pd.DataFrame(fixed, columns=fields)
    df = df.replace(NA_TOKENS, np.nan)

    for col in ("ts", "duration", "orig_bytes", "resp_bytes",
                "orig_pkts", "resp_pkts", "id.resp_p", "id.orig_p"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    log.info("read %s: %d rows, %d columns", path.name, len(df), len(df.columns))
    return df


# --------------------------------------------------------------------------
# CTU-13: labelled bidirectional NetFlow (Argus)
# --------------------------------------------------------------------------
# CTU-13 ships `detailed-bidirectional-flow-labels/*.binetflow`, which carry
# their own labels. These are connection-level summaries exactly like a Zeek
# conn.log, so no Zeek installation and no packet captures are required -- a
# 369 MB flow file replaces a 1.2 GB pcap plus a Zeek build.
#
# One field genuinely differs: Argus records no application-protocol
# annotation, so `service` is absent and is imputed as unknown downstream.
# Every field the interval derivation needs -- start time, endpoints, duration,
# bytes, packets, state -- is present in both corpora.
ARGUS_TO_ZEEK = {
    "StartTime": "ts",
    "Dur":       "duration",
    "Proto":     "proto",
    "SrcAddr":   "id.orig_h",
    "Sport":     "id.orig_p",
    "DstAddr":   "id.resp_h",
    "Dport":     "id.resp_p",
    "State":     "conn_state",
    "SrcBytes":  "orig_bytes",
    "Label":     "label",
}


def read_binetflow(path: Path, chunksize: int | None = 500_000) -> pd.DataFrame:
    """Read a CTU-13 .binetflow into the same schema as a Zeek conn.log.

    Read in chunks and filtered as we go: these files run to hundreds of MB and
    the overwhelming majority of rows are Background traffic this study drops
    anyway, so filtering during the read keeps peak memory low.
    """
    path = Path(path)
    keep = []
    reader_iter = pd.read_csv(path, chunksize=chunksize, low_memory=False)

    for chunk in reader_iter:
        cols = {c: c.strip() for c in chunk.columns}
        chunk = chunk.rename(columns=cols)
        if "Label" not in chunk.columns:
            raise ValueError(f"{path}: no Label column; is this a binetflow?")
        lab = chunk["Label"].astype(str).str.lower()
        # drop Background here -- it is the bulk of the file and is excluded by
        # the study design anyway (Section III-D)
        drop = lab.str.contains("|".join(C.EXCLUDE_LABEL_TOKENS), na=False)
        keep.append(chunk[~drop])

    if not keep:
        return pd.DataFrame()
    df = pd.concat(keep, ignore_index=True)

    out = pd.DataFrame()
    for src, dst in ARGUS_TO_ZEEK.items():
        if src in df.columns:
            out[dst] = df[src]

    # Argus reports total bytes/packets; derive the responder side
    tot_b = pd.to_numeric(df.get("TotBytes"), errors="coerce")
    src_b = pd.to_numeric(df.get("SrcBytes"), errors="coerce")
    out["resp_bytes"] = (tot_b - src_b).clip(lower=0)
    if "SrcPkts" in df.columns and "DstPkts" in df.columns:
        out["orig_pkts"] = pd.to_numeric(df["SrcPkts"], errors="coerce")
        out["resp_pkts"] = pd.to_numeric(df["DstPkts"], errors="coerce")
    else:
        tot_p = pd.to_numeric(df.get("TotPkts"), errors="coerce")
        out["orig_pkts"] = tot_p
        out["resp_pkts"] = np.nan          # not recorded separately by Argus
    out["service"] = np.nan                # Argus records no application label

    # StartTime is a timestamp string; the derivation needs epoch SECONDS.
    #
    # Do NOT use .astype("int64") here. pandas 2.x returns datetime64[ns] and
    # pandas 3.x returns datetime64[us], so the integer representation means
    # different things on different versions -- the periods would come out
    # 1000x wrong on one of them, silently, with no error. total_seconds() is
    # unit-independent. There is a regression test for this.
    _t = pd.to_datetime(out["ts"], errors="coerce")
    out["ts"] = (_t - pd.Timestamp("1970-01-01")).dt.total_seconds()
    for c in ("duration", "orig_bytes", "id.resp_p"):
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    out["proto"] = out["proto"].astype(str).str.lower()

    out = out.dropna(subset=["ts"]).reset_index(drop=True)
    log.info("read %s: %d non-background flows", path.name, len(out))
    return out


def read_any(path: Path) -> pd.DataFrame:
    """Dispatch on file type: Zeek conn.log (IoT-23) or Argus binetflow (CTU-13)."""
    path = Path(path)
    if path.suffix == ".binetflow" or "binetflow" in path.name:
        return read_binetflow(path)
    return read_conn_log(path)


def zeek_replay(pcap: Path, out_dir: Path, zeek_bin: str = "zeek") -> Path:
    """Regenerate conn.log from a packet capture (used for CTU-13).

    Returns the path to the produced conn.log.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [zeek_bin, "-r", str(pcap), "LogAscii::use_json=F"]
    log.info("zeek replay: %s", " ".join(cmd))
    subprocess.run(cmd, cwd=out_dir, check=True)
    conn = out_dir / "conn.log"
    if not conn.exists():
        raise FileNotFoundError(f"zeek produced no conn.log in {out_dir}")
    return conn


# --------------------------------------------------------------------------
# Class assignment
# --------------------------------------------------------------------------
def _label_column(df: pd.DataFrame) -> str | None:
    for cand in ("label", "detailed-label", "tunnel_parents   label   detailed-label"):
        if cand in df.columns:
            return cand
    # IoT-23's trailing column is often unnamed after the split
    tail = df.columns[-1]
    if df[tail].astype(str).str.contains("C&C|Benign", case=False, na=False).any():
        return tail
    return None


def assign_class(df: pd.DataFrame) -> pd.DataFrame:
    """Add a ``y`` column: 1 = C2, 0 = benign, NaN = excluded.

    Background traffic of uncertain provenance is dropped rather than assumed
    benign -- it would contaminate the false-alarm denominator (Section III-D).
    """
    col = _label_column(df)
    if col is None:
        raise ValueError("no label column found; is this a labelled corpus?")

    lab = df[col].astype(str).str.lower()
    y = pd.Series(np.nan, index=df.index, dtype="float")

    excluded = lab.apply(lambda s: any(t in s for t in C.EXCLUDE_LABEL_TOKENS))
    is_c2 = lab.apply(lambda s: any(t in s for t in C.C2_LABEL_TOKENS))
    is_benign = lab.apply(lambda s: any(t in s for t in C.BENIGN_LABEL_TOKENS))

    y[is_c2 & ~excluded] = 1.0
    y[is_benign & ~excluded & y.isna()] = 0.0

    out = df.copy()
    out["y"] = y
    n_drop = int(out["y"].isna().sum())
    log.info("class assignment: %d C2, %d benign, %d excluded",
             int((y == 1).sum()), int((y == 0).sum()), n_drop)
    return out


# --------------------------------------------------------------------------
# Channel construction
# --------------------------------------------------------------------------
def build_channels(df: pd.DataFrame) -> pd.DataFrame:
    """Attach a ``channel_id`` identifying the communication relationship.

    Eq. (1) in the paper. A channel is the set of flows sharing source address,
    destination address, destination port and transport protocol.
    """
    missing = [k for k in C.CHANNEL_KEY if k not in df.columns]
    if missing:
        raise ValueError(f"missing channel key columns: {missing}")

    out = df.copy()
    key = out[list(C.CHANNEL_KEY)].astype(str).agg("|".join, axis=1)
    out["channel_id"] = key
    log.info("built %d channels over %d flows", key.nunique(), len(out))
    return out


def filter_min_observations(df: pd.DataFrame,
                            min_obs: int = C.MIN_OBSERVATIONS
                            ) -> tuple[pd.DataFrame, dict]:
    """Drop channels seen fewer than ``min_obs`` times.

    Returns the filtered frame and a report the paper is obliged to publish
    (Section III-D promises the excluded count).
    """
    counts = df["channel_id"].value_counts()
    keep = counts[counts >= min_obs].index
    kept = df[df["channel_id"].isin(keep)].copy()
    report = {
        "channels_before": int(counts.size),
        "channels_kept": int(len(keep)),
        "channels_dropped": int(counts.size - len(keep)),
        "flows_before": int(len(df)),
        "flows_kept": int(len(kept)),
        "min_observations": int(min_obs),
    }
    log.info("min-observation filter: kept %d/%d channels",
             report["channels_kept"], report["channels_before"])
    return kept, report


def prepare(paths: list[Path]) -> tuple[pd.DataFrame, dict]:
    """Full stage B: read -> class -> channels -> min-observation filter.

    Classes are assigned PER FILE, before concatenation. This matters when the
    two corpora are mixed: IoT-23 carries its label in a compound Zeek column
    while CTU-13 names it ``label``, so a concatenated frame holds both columns
    with NaN in the rows from the other corpus. Assigning first, concatenating
    second, means whichever column a file uses is the one read for that file --
    otherwise one corpus is silently dropped as unlabelled.
    """
    frames = []
    for p in paths:
        p = Path(p)
        is_ctu13 = p.suffix == ".binetflow" or "binetflow" in p.name
        d = read_any(p)
        d = assign_class(d)
        d["source_capture"] = p.stem
        # Tagged by dispatch, not by filename pattern-matching a second time --
        # this is what the cross-corpus generalisation check (generalize.py)
        # splits train/test on, so it needs to be right regardless of what a
        # user names their downloaded files.
        d["corpus"] = "ctu13" if is_ctu13 else "iot23"
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=["y"]).reset_index(drop=True)
    df["y"] = df["y"].astype(int)
    df = build_channels(df)
    return filter_min_observations(df)

In [ ]:
%%writefile src/intervals.py
"""Stage C -- beacon-interval derivation and stratification.

This is the contribution of the paper. Neither IoT-23 nor CTU-13 records a
beacon period; this module computes one per channel from connection timestamps
and assigns each channel to an interval band, converting beacon interval from
an unrecorded property of the data into an explicit experimental variable.

Implements Eq. (2)-(4) of the manuscript.
"""
from __future__ import annotations

import logging

import numpy as np
import pandas as pd

from . import config as C

log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# Core statistics
# --------------------------------------------------------------------------
def inter_arrival(ts: np.ndarray) -> np.ndarray:
    """Eq. (2): sorted connection timestamps differenced to a Delta-t series."""
    t = np.sort(np.asarray(ts, dtype=float))
    return np.diff(t)


def median_absolute_deviation(x: np.ndarray) -> float:
    """Eq. (3): MAD = median(|x - median(x)|).

    Median and MAD are used in preference to mean and standard deviation
    because beacon series carry heavy right tails from retries, missed
    check-ins and network delay; a mean is dragged upward by a handful of long
    gaps and misrepresents the operator-configured period.
    """
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return float("nan")
    return float(np.median(np.abs(x - np.median(x))))


def channel_statistics(ts: np.ndarray) -> dict:
    """Per-channel timing summary. Returns NaNs when too few observations."""
    dt = inter_arrival(ts)
    if dt.size == 0:
        return dict(n_flows=int(np.size(ts)), dt_mean=np.nan, dt_median=np.nan,
                    dt_std=np.nan, dt_mad=np.nan, jitter_ratio=np.nan,
                    period=np.nan)
    period = float(np.median(dt))                     # P-tilde
    mad = median_absolute_deviation(dt)
    # Eq. (4). Guard against a degenerate zero period (duplicate timestamps).
    jitter = float(mad / period) if period > 0 else np.nan
    return dict(
        n_flows=int(np.size(ts)),
        dt_mean=float(np.mean(dt)),
        dt_median=period,
        dt_std=float(np.std(dt, ddof=1)) if dt.size > 1 else 0.0,
        dt_mad=mad,
        jitter_ratio=jitter,
        period=period,
    )


# --------------------------------------------------------------------------
# Banding
# --------------------------------------------------------------------------
def assign_band(period: float, edges: tuple[float, float] = C.BAND_EDGES_S) -> str:
    """Map a characteristic period to B1 / B2 / B3.

    Boundaries are fixed in config before any detector runs.
    """
    lo, hi = edges
    if not np.isfinite(period):
        return "unknown"
    if period < lo:
        return "B1"
    if period < hi:
        return "B2"
    return "B3"


def derive(df: pd.DataFrame, ts_col: str = "ts") -> tuple[pd.DataFrame, pd.DataFrame]:
    """Derive per-channel periods and broadcast band labels back to flows.

    Returns
    -------
    flows : the input frame with timing features and a ``band`` column added
    channels : one row per channel -- the artefact released alongside the paper
    """
    if ts_col not in df.columns:
        raise ValueError(f"missing timestamp column {ts_col!r}")

    recs = []
    for cid, g in df.groupby("channel_id", sort=False):
        st = channel_statistics(g[ts_col].to_numpy())
        st["channel_id"] = cid
        st["band"] = assign_band(st["period"])
        # A channel is positive if any of its flows is labelled C2. In these
        # corpora a channel is homogeneous in practice; this is defensive.
        st["y_channel"] = int(g["y"].max()) if "y" in g.columns else -1
        recs.append(st)

    channels = pd.DataFrame.from_records(recs)

    # "period" is carried onto flows alongside the discrete band. The model
    # never sees it (features.py enforces that), but the continuous value is
    # what a fine-grained, band-free analysis needs -- see src/generalize.py.
    cols = ["channel_id", "band", "period"] + C.TIMING_FEATURES
    flows = df.merge(channels[cols], on="channel_id", how="left")

    unknown = int((flows["band"] == "unknown").sum())
    if unknown:
        log.warning("%d flows in channels with no estimable period -- dropped", unknown)
        flows = flows[flows["band"] != "unknown"].reset_index(drop=True)
        channels = channels[channels["band"] != "unknown"].reset_index(drop=True)

    log.info("band populations (channels): %s",
             channels["band"].value_counts().to_dict())
    return flows, channels


def band_report(channels: pd.DataFrame) -> dict:
    """Populations per band -- the paper is obliged to report these."""
    rep = {}
    for b in C.BANDS:
        sub = channels[channels["band"] == b]
        rep[b] = {
            "channels": int(len(sub)),
            "c2_channels": int((sub["y_channel"] == 1).sum()) if len(sub) else 0,
            "median_period_s": float(sub["period"].median()) if len(sub) else float("nan"),
            "median_jitter_ratio": float(sub["jitter_ratio"].median()) if len(sub) else float("nan"),
        }
    return rep

In [ ]:
%%writefile src/features.py
"""Stage D -- feature engineering.

Builds the matrix consumed by the learning baselines. Two invariants are
enforced here rather than trusted:

1. The band label never enters the feature matrix. It is a stratifier, and
   feeding it in would leak the quantity the study measures.
2. Scaler and encoder are FITTED ON TRAINING FOLDS ONLY. They are returned so
   the caller applies them, unchanged, to held-out data.
"""
from __future__ import annotations

import logging

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from . import config as C

log = logging.getLogger(__name__)


def feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    num = [c for c in C.ZEEK_NUMERIC + C.TIMING_FEATURES if c in df.columns]
    cat = [c for c in C.ZEEK_CATEGORICAL if c in df.columns]
    leaked = (set(num) | set(cat)) & C.FORBIDDEN_FEATURES
    if leaked:
        raise AssertionError(f"forbidden feature(s) in matrix: {sorted(leaked)}")
    return num, cat


def build_transformer(num: list[str], cat: list[str]) -> ColumnTransformer:
    """Min-max scaling for numerics, one-hot for categoricals.

    Min-max is used because IoT-23's numeric features are strongly skewed and
    the distance-based classifier is sensitive to feature scale.
    """
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
    ])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        [("num", num_pipe, num), ("cat", cat_pipe, cat)],
        remainder="drop",
    )


def assemble(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """Return (X_raw, y, groups, bands).

    X_raw is left untransformed on purpose -- fitting happens inside the CV
    loop so that no held-out information reaches the scaler.
    """
    num, cat = feature_columns(df)
    X = df[num + cat].copy()
    y = df["y"].to_numpy(dtype=int)
    groups = df["channel_id"].to_numpy()
    bands = df["band"].to_numpy()
    log.info("feature matrix: %d rows, %d numeric, %d categorical",
             len(X), len(num), len(cat))
    return X, y, groups, bands


def assemble_ext(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, np.ndarray,
                                             np.ndarray, np.ndarray, np.ndarray]:
    """Like ``assemble``, plus the two columns the extension analyses need.

    Returns (X_raw, y, groups, bands, periods, corpus). ``periods`` is the
    continuous per-channel beacon period (never a feature -- see
    FORBIDDEN_FEATURES); ``corpus`` is "iot23" or "ctu13", used only to split
    train/test in the cross-corpus generalisation check, never as a feature.
    """
    X, y, groups, bands = assemble(df)
    if "period" not in df.columns:
        raise ValueError("no 'period' column; did you run intervals.derive()?")
    if "corpus" not in df.columns:
        raise ValueError("no 'corpus' column; did you run preprocess.prepare()?")
    periods = df["period"].to_numpy(dtype=float)
    corpus = df["corpus"].to_numpy()
    return X, y, groups, bands, periods, corpus


def feature_names(ct: ColumnTransformer) -> list[str]:
    """Human-readable names after fitting, for the importance figure."""
    try:
        return list(ct.get_feature_names_out())
    except Exception:                                   # pragma: no cover
        return [f"f{i}" for i in range(ct.transformers_[0][1][-1].n_features_in_)]

In [ ]:
%%writefile src/models.py
"""Stage E -- learning baselines with leakage-safe cross-validation.

The two methodological commitments the paper makes are enforced in code here,
not left to convention:

* Partitioning uses GroupKFold keyed on ``channel_id``. Flows from one
  beaconing channel are near-duplicates; a random flow-level split would leak
  and inflate every reported score.
* SMOTE is applied INSIDE training folds only. Resampling before partitioning
  synthesises minority points from observations that later appear in the
  evaluation fold -- a well-known source of optimistic bias.

Both are asserted at runtime.
"""
from __future__ import annotations

import logging
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.neighbors import KNeighborsClassifier

from . import config as C
from .features import build_transformer, feature_columns, feature_names

log = logging.getLogger(__name__)

try:
    from imblearn.over_sampling import SMOTE
    HAVE_SMOTE = True
except ImportError:                                     # pragma: no cover
    HAVE_SMOTE = False
    warnings.warn(
        "imbalanced-learn not installed: falling back to class weighting. "
        "Install it (pip install imbalanced-learn) to reproduce the paper's "
        "stated methodology.", RuntimeWarning)


def _make_model(name: str, params: dict, seed: int):
    if name == "Random Forest":
        return RandomForestClassifier(random_state=seed, **{**C.RF_PARAMS, **params})
    if name == "k-NN":
        return KNeighborsClassifier(**{**C.KNN_PARAMS, **params})
    if name == "Gradient Boosting":
        return HistGradientBoostingClassifier(random_state=seed, **{**C.GB_PARAMS, **params})
    raise ValueError(name)


def _resample(Xt: np.ndarray, y: np.ndarray, seed: int):
    """SMOTE on a TRAINING fold. No-op if the minority class is too small."""
    if not (C.USE_SMOTE and HAVE_SMOTE):
        return Xt, y
    n_min = int(min(np.bincount(y)))
    if n_min < 6:
        log.warning("minority class n=%d too small for SMOTE; skipping", n_min)
        return Xt, y
    sm = SMOTE(random_state=seed, k_neighbors=min(5, n_min - 1))
    return sm.fit_resample(Xt, y)


def _assert_no_group_overlap(groups, tr, te) -> None:
    overlap = set(groups[tr]) & set(groups[te])
    if overlap:
        raise AssertionError(
            f"channel leakage: {len(overlap)} channel(s) in both train and test")


def _select_params(name: str, grid: dict, Xtr: np.ndarray, ytr: np.ndarray,
                   gtr: np.ndarray, seed: int) -> dict:
    """Inner GroupKFold grid search, factored out so cross_validate and
    cross_corpus_eval (generalize.py's caller) tune hyperparameters the same
    way -- always on the training split only, never touching held-out data."""
    best_score, best_params = -np.inf, None
    inner = GroupKFold(n_splits=3)
    for params in ParameterGrid(grid):
        scores = []
        for itr, ite in inner.split(Xtr, ytr, groups=gtr):
            Xi, yi = _resample(Xtr[itr], ytr[itr], seed)
            m = _make_model(name, params, seed).fit(Xi, yi)
            p = m.predict_proba(Xtr[ite])[:, 1]
            pred = (p >= C.DECISION_THRESHOLD).astype(int)
            tp = int(((pred == 1) & (ytr[ite] == 1)).sum())
            fp = int(((pred == 1) & (ytr[ite] == 0)).sum())
            fn = int(((pred == 0) & (ytr[ite] == 1)).sum())
            f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
            scores.append(f1)
        mean = float(np.mean(scores))
        if mean > best_score:
            best_score, best_params = mean, params
    return best_params


def cross_validate(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                   bands: np.ndarray, seeds=C.RANDOM_SEEDS,
                   periods: np.ndarray | None = None,
                   corpus: np.ndarray | None = None) -> dict:
    """Run the full protocol and return out-of-fold predictions per model.

    Returns
    -------
    {model: {"y_true", "y_score", "band", "seed", [+ "period", "corpus"]}}
    Out-of-fold predictions are concatenated across folds, so every flow is
    predicted exactly once per seed by a model that never saw its channel.

    ``periods`` and ``corpus``, when given, are threaded through unchanged --
    like ``bands``, they never reach the model (features.py enforces that);
    they are carried only so the fine-grained dose-response analysis
    (generalize.py) can re-slice these same predictions without retraining.
    """
    num, cat = feature_columns(X)
    grids = {"Random Forest": C.RF_GRID, "k-NN": C.KNN_GRID, "Gradient Boosting": C.GB_GRID}
    keys = ["y_true", "y_score", "band", "seed"]
    if periods is not None:
        keys.append("period")
    if corpus is not None:
        keys.append("corpus")
    out = {m: {k: [] for k in keys} for m in grids}
    chosen: dict[str, list] = {m: [] for m in grids}

    for seed in seeds:
        gkf = GroupKFold(n_splits=C.N_FOLDS)
        for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups)):
            _assert_no_group_overlap(groups, tr, te)

            ct = build_transformer(num, cat)
            Xtr = ct.fit_transform(X.iloc[tr])       # FIT on training only
            Xte = ct.transform(X.iloc[te])           # transform held-out
            ytr, yte = y[tr], y[te]

            Xtr_r, ytr_r = _resample(Xtr, ytr, seed)

            for name, grid in grids.items():
                best_params = _select_params(name, grid, Xtr, ytr, groups[tr], seed)
                model = _make_model(name, best_params, seed).fit(Xtr_r, ytr_r)
                proba = model.predict_proba(Xte)[:, 1]

                out[name]["y_true"].append(yte)
                out[name]["y_score"].append(proba)
                out[name]["band"].append(bands[te])
                out[name]["seed"].append(np.full(te.size, seed))
                if periods is not None:
                    out[name]["period"].append(periods[te])
                if corpus is not None:
                    out[name]["corpus"].append(corpus[te])
                chosen[name].append(best_params)

            log.info("seed %d fold %d/%d done", seed, fold + 1, C.N_FOLDS)

    for m in out:
        out[m] = {k: np.concatenate(v) for k, v in out[m].items()}
        out[m]["chosen_params"] = chosen[m]
    return out


def cross_corpus_eval(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                      corpus: np.ndarray, seeds=C.RANDOM_SEEDS,
                      bands: np.ndarray | None = None,
                      periods: np.ndarray | None = None) -> dict:
    """Train on one corpus in full, evaluate cold on the other. Both directions.

    This is NOT cross-validation -- there is exactly one split per direction,
    fixed by which corpus is which, so "seeds" here vary only the model's own
    stochasticity (RF's bootstrap, SMOTE's neighbour sampling), not the split.
    GroupKFold is still used, but only internally, to select hyperparameters
    on the training corpus without ever touching the test corpus.

    Returns
    -------
    {"iot23_to_ctu13": {model: {...}}, "ctu13_to_iot23": {model: {...}}}
    Each inner dict has the same shape as one seed's slice of cross_validate's
    output: y_true, y_score, band, period, seed.
    """
    num, cat = feature_columns(X)
    grids = {"Random Forest": C.RF_GRID, "k-NN": C.KNN_GRID, "Gradient Boosting": C.GB_GRID}
    directions = [("iot23", "ctu13"), ("ctu13", "iot23")]
    results: dict[str, dict] = {}

    for train_corpus, test_corpus in directions:
        tr = np.flatnonzero(corpus == train_corpus)
        te = np.flatnonzero(corpus == test_corpus)
        if tr.size == 0 or te.size == 0:
            log.warning("cross_corpus_eval: no data for %s -> %s (skipped)",
                        train_corpus, test_corpus)
            continue
        # A group can only ever appear on one side of a corpus split by
        # construction (a channel's five-tuple belongs to one capture file),
        # but assert it anyway -- the same invariant cross_validate enforces.
        _assert_no_group_overlap(groups, tr, te)

        key = f"{train_corpus}_to_{test_corpus}"
        keys = ["y_true", "y_score", "seed"]
        if bands is not None:
            keys.append("band")
        if periods is not None:
            keys.append("period")
        out = {m: {k: [] for k in keys} for m in grids}

        for seed in seeds:
            ct = build_transformer(num, cat)
            Xtr = ct.fit_transform(X.iloc[tr])
            Xte = ct.transform(X.iloc[te])
            ytr, yte = y[tr], y[te]
            Xtr_r, ytr_r = _resample(Xtr, ytr, seed)

            for name, grid in grids.items():
                best_params = _select_params(name, grid, Xtr, ytr, groups[tr], seed)
                model = _make_model(name, best_params, seed).fit(Xtr_r, ytr_r)
                proba = model.predict_proba(Xte)[:, 1]
                out[name]["y_true"].append(yte)
                out[name]["y_score"].append(proba)
                out[name]["seed"].append(np.full(te.size, seed))
                if bands is not None:
                    out[name]["band"].append(bands[te])
                if periods is not None:
                    out[name]["period"].append(periods[te])

            log.info("cross-corpus %s: seed %d done", key, seed)

        for m in out:
            out[m] = {k: np.concatenate(v) for k, v in out[m].items()}
        results[key] = out

    return results


def fit_final_and_importance(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                             seed: int = C.RANDOM_SEEDS[0],
                             n_repeats: int = 5) -> dict:
    """Fit Random Forest on all data and compute permutation importance.

    Importance is measured on a held-out split (still grouped by channel), not
    on training data -- importances computed in-sample are optimistic.
    """
    num, cat = feature_columns(X)
    gkf = GroupKFold(n_splits=C.N_FOLDS)
    tr, te = next(gkf.split(X, y, groups=groups))
    _assert_no_group_overlap(groups, tr, te)

    ct = build_transformer(num, cat)
    Xtr = ct.fit_transform(X.iloc[tr])
    Xte = ct.transform(X.iloc[te])
    Xtr_r, ytr_r = _resample(Xtr, y[tr], seed)

    rf = _make_model("Random Forest", {"max_features": "sqrt"}, seed).fit(Xtr_r, ytr_r)
    pi = permutation_importance(rf, Xte, y[te], n_repeats=n_repeats,
                                random_state=seed, scoring="f1", n_jobs=-1)
    names = feature_names(ct)
    imp = {names[i]: (float(pi.importances_mean[i]), float(pi.importances_std[i]))
           for i in range(len(names))}
    return {"importance": imp, "model": rf, "transformer": ct}


def rf_oob_curve(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                 seed: int = C.RANDOM_SEEDS[0],
                 grid=(25, 50, 100, 150, 200, 250, 300)) -> tuple[np.ndarray, np.ndarray]:
    """Out-of-bag error against forest size.

    Random Forest has no loss curve; this is the honest equivalent.
    """
    num, cat = feature_columns(X)
    ct = build_transformer(num, cat)
    Xt = ct.fit_transform(X)
    ns, errs = [], []
    for n in grid:
        rf = RandomForestClassifier(
            n_estimators=int(n), oob_score=True, bootstrap=True,
            random_state=seed, n_jobs=-1,
            criterion=C.RF_PARAMS["criterion"],
            class_weight=C.RF_PARAMS["class_weight"])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rf.fit(Xt, y)
        ns.append(int(n))
        errs.append(1.0 - float(rf.oob_score_))
    return np.array(ns), np.array(errs)


def knn_k_sweep(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                seed: int = C.RANDOM_SEEDS[0]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Cross-validated F1 against k. The honest equivalent of a training curve."""
    num, cat = feature_columns(X)
    ks = np.array(C.KNN_GRID["n_neighbors"])
    means, stds = [], []
    gkf = GroupKFold(n_splits=C.N_FOLDS)
    for k in ks:
        fold_scores = []
        for tr, te in gkf.split(X, y, groups=groups):
            ct = build_transformer(num, cat)
            Xtr = ct.fit_transform(X.iloc[tr])
            Xte = ct.transform(X.iloc[te])
            Xtr_r, ytr_r = _resample(Xtr, y[tr], seed)
            m = _make_model("k-NN", {"n_neighbors": int(k)}, seed).fit(Xtr_r, ytr_r)
            pred = (m.predict_proba(Xte)[:, 1] >= C.DECISION_THRESHOLD).astype(int)
            tp = int(((pred == 1) & (y[te] == 1)).sum())
            fp = int(((pred == 1) & (y[te] == 0)).sum())
            fn = int(((pred == 0) & (y[te] == 1)).sum())
            fold_scores.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0)
        means.append(float(np.mean(fold_scores)))
        stds.append(float(np.std(fold_scores)))
    return ks, np.array(means), np.array(stds)

In [ ]:
%%writefile src/evaluate.py
"""Stage F -- per-band evaluation, the breakdown criterion, and LaTeX output.

Every metric is computed WITHIN band. A detector that performs well on
sub-minute beacons and poorly on ten-minute beacons produces a respectable
pooled score, and pooling is exactly what conceals the failure this study
exists to locate.

Implements Eq. (7)-(12).
"""
from __future__ import annotations

import json
import logging
from pathlib import Path

import numpy as np

from . import config as C

log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# Metrics -- Eq. (7)-(11)
# --------------------------------------------------------------------------
def confusion(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[int, int, int, int]:
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return tp, fp, tn, fn


def metrics(y_true: np.ndarray, y_score: np.ndarray,
            threshold: float = C.DECISION_THRESHOLD) -> dict:
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tp, fp, tn, fn = confusion(np.asarray(y_true), y_pred)
    dr = tp / (tp + fn) if (tp + fn) else float("nan")          # DR, FAR: Eq. 7-8
    far = fp / (fp + tn) if (fp + tn) else float("nan")
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = dr if np.isfinite(dr) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0   # F1: Eq. 8
    return dict(DR=dr, FAR=far, P=prec, R=rec, F1=f1,
                tp=tp, fp=fp, tn=tn, fn=fn,
                n=int(len(y_true)), n_pos=int(np.sum(y_true)))


def per_band(pred: dict) -> dict:
    """Metrics per band, aggregated over seeds as mean +/- sd."""
    out: dict[str, dict] = {}
    seeds = np.unique(pred["seed"])
    for b in C.BANDS:
        mask_b = pred["band"] == b
        if not mask_b.any():
            continue
        per_seed = []
        for s in seeds:
            m = mask_b & (pred["seed"] == s)
            if m.sum() == 0:
                continue
            per_seed.append(metrics(pred["y_true"][m], pred["y_score"][m]))
        if not per_seed:
            continue
        agg = {}
        for k in ("DR", "FAR", "P", "R", "F1"):
            vals = np.array([d[k] for d in per_seed], dtype=float)
            agg[k] = float(np.nanmean(vals))
            agg[k + "_sd"] = float(np.nanstd(vals))
        agg["n"] = int(mask_b.sum() // max(len(seeds), 1))
        agg["n_pos"] = int(per_seed[0]["n_pos"])
        out[b] = agg
    return out


def breakdown(band_metrics: dict, tau: float = C.TAU, delta: float = C.DELTA) -> dict:
    """Eq. (9) in the paper. A detector breaks down in band B if DR(B) < tau, or if
    DR(B1) - DR(B) > delta. Both constants are pre-registered in config.py."""
    if "B1" not in band_metrics:
        return {}
    base = band_metrics["B1"]["DR"]
    res = {}
    for b, m in band_metrics.items():
        res[b] = {
            "DR": m["DR"],
            "below_tau": bool(m["DR"] < tau),
            "drop_exceeds_delta": bool((base - m["DR"]) > delta),
        }
        res[b]["breaks_down"] = res[b]["below_tau"] or res[b]["drop_exceeds_delta"]
    return res


def evaluate_all(predictions: dict, suricata: dict | None = None) -> dict:
    """Assemble the full results object the paper and figures consume."""
    table = {}
    for model, pred in predictions.items():
        table[model] = per_band(pred)
    if suricata:
        table["Suricata"] = per_band(suricata)
    return {
        "per_band": table,
        "breakdown": {m: breakdown(v) for m, v in table.items()},
        "config": {"tau": C.TAU, "delta": C.DELTA,
                   "threshold": C.DECISION_THRESHOLD,
                   "band_edges_s": list(C.BAND_EDGES_S),
                   "seeds": list(C.RANDOM_SEEDS),
                   "n_folds": C.N_FOLDS},
    }


# --------------------------------------------------------------------------
# LaTeX output -- table and \newcommand macros the manuscript reads
# --------------------------------------------------------------------------
def write_latex_table(results: dict, path: Path) -> None:
    """Per-band table with the best value in each column bolded."""
    tbl = results["per_band"]
    models = [m for m in C.DETECTORS if m in tbl]
    lines = [
        r"% Generated by src/evaluate.py -- regenerate, do not hand-edit.",
        r"\begin{table}[!t]",
        r"\renewcommand{\arraystretch}{1.15}",
        r"\caption{Per-Band Detection Performance, Mean over Five Seeds. Best "
        r"Value in Each Column Within Each Band in \textbf{Bold}. Detection "
        r"rate is recall, so it is not repeated as a separate column.}",
        r"\label{tab:perband}",
        r"\centering",
        r"\small",
        r"\begin{tabular}{llcccc}",
        r"\hline",
        r"Band & Detector & DR & FAR (\%) & Prec. & F1\\",
        r"\hline",
    ]
    for b in C.BANDS:
        present = [m for m in models if b in tbl[m]]
        if not present:
            continue
        best = {}
        for key in ("DR", "FAR", "P", "F1"):
            vals = {m: tbl[m][b][key] for m in present}
            best[key] = (min if key == "FAR" else max)(vals, key=vals.get)
        for i, m in enumerate(present):
            v = tbl[m][b]
            cells = []
            for key in ("DR", "FAR", "P", "F1"):
                x = v[key] * 100 if key == "FAR" else v[key]
                txt = f"{x:.2f}" if key == "FAR" else f"{x:.3f}"
                cells.append(r"\textbf{" + txt + "}" if best[key] == m else txt)
            head = f"{b} ($n$={v['n']})" if i == 0 else ""
            lines.append(f"{head} & {m} & " + " & ".join(cells) + r"\\")
        lines.append(r"\hline")
    lines += [r"\end{tabular}", r"\end{table}"]
    Path(path).write_text("\n".join(lines), encoding="utf-8")
    log.info("wrote %s", path)


def write_macros(results: dict, band_rep: dict, filt_rep: dict, path: Path) -> None:
    """Emit \\newcommand macros so the prose never hard-codes a number.

    The manuscript writes \\ResDRrfBthree instead of a literal, so every number
    in the text is guaranteed to match the run that produced it.
    """
    def cmd(name, value):
        return r"\newcommand{\%s}{%s}" % (name, value)

    idx = {"B1": "Bone", "B2": "Btwo", "B3": "Bthree"}
    mdl = {"Suricata": "sur", "Random Forest": "rf", "Gradient Boosting": "gb", "k-NN": "knn"}
    out = [r"% Generated by src/evaluate.py -- regenerate, do not hand-edit."]

    for m, key in mdl.items():
        if m not in results["per_band"]:
            continue
        for b, bkey in idx.items():
            if b not in results["per_band"][m]:
                continue
            v = results["per_band"][m][b]
            out.append(cmd(f"ResDR{key}{bkey}", f"{v['DR']:.3f}"))
            out.append(cmd(f"ResFAR{key}{bkey}", f"{v['FAR']*100:.2f}"))
            out.append(cmd(f"ResFone{key}{bkey}", f"{v['F1']:.3f}"))
            out.append(cmd(f"ResDRsd{key}{bkey}", f"{v['DR_sd']:.3f}"))

    for b, bkey in idx.items():
        if b in band_rep:
            out.append(cmd(f"NChan{bkey}", f"{band_rep[b]['channels']:,}"))
            out.append(cmd(f"NCtwo{bkey}", f"{band_rep[b]['c2_channels']:,}"))
            p = band_rep[b]["median_period_s"]
            out.append(cmd(f"MedP{bkey}", "--" if not np.isfinite(p) else f"{p:.0f}"))

    out.append(cmd("NChannelsKept", f"{filt_rep.get('channels_kept', 0):,}"))
    out.append(cmd("NChannelsDropped", f"{filt_rep.get('channels_dropped', 0):,}"))
    out.append(cmd("NFlowsKept", f"{filt_rep.get('flows_kept', 0):,}"))
    out.append(cmd("TauVal", f"{C.TAU:.2f}"))
    out.append(cmd("DeltaVal", f"{C.DELTA:.2f}"))

    Path(path).write_text("\n".join(out) + "\n", encoding="utf-8")
    log.info("wrote %s", path)


def save_json(obj: dict, path: Path) -> None:
    Path(path).write_text(json.dumps(obj, indent=2, default=float), encoding="utf-8")
    log.info("wrote %s", path)

In [ ]:
%%writefile src/generalize.py
"""Extension analyses: a continuous dose-response curve over beacon interval,
and cross-corpus generalisation -- neither reported anywhere in this study's
own Related Work section.

Both re-use artefacts the main pipeline already produces:

* The dose-response curve re-slices the SAME out-of-fold predictions
  ``models.cross_validate`` already makes, at a finer resolution than the
  three pre-registered bands. No retraining -- ``period`` (continuous, never
  a model feature) travels alongside each prediction precisely so this module
  can bin it however it likes after the fact.
* Cross-corpus generalisation needs one new pass (``models.cross_corpus_eval``):
  train on one corpus in full, evaluate cold on the other, both directions.

Both are exploratory in the sense that pre-registration (Section III-I of the
paper) does not apply to them -- they were not fixed before this run's
results were seen, because they did not exist yet. Report them as such.
"""
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np

from . import config as C
from .evaluate import metrics, per_band, save_json

log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# Fine-grained binning
# --------------------------------------------------------------------------
def fine_bin_edges(channel_periods: np.ndarray, n_bins: int = 8) -> np.ndarray:
    """Quantile-spaced edges (in log-period) over real channel periods.

    Quantile spacing rather than fixed log-spacing, because real beacon
    periods cluster around a handful of operator-configured defaults instead
    of filling the interval range evenly -- fixed spacing would leave some
    bins nearly empty and others overcrowded. Duplicate quantiles (repeated
    period values collapsing a bin edge) are dropped rather than padded.
    """
    p = np.asarray(channel_periods, dtype=float)
    p = p[np.isfinite(p) & (p > 0)]
    if p.size < n_bins * 2:
        raise ValueError(
            f"only {p.size} channels available for {n_bins} bins "
            f"(need >= {n_bins * 2}); reduce n_bins")
    logp = np.log(p)
    edges = np.unique(np.quantile(logp, np.linspace(0, 1, n_bins + 1)))
    if edges.size < 3:
        raise ValueError(
            "period distribution too concentrated for quantile binning at "
            f"n_bins={n_bins}; reduce n_bins")
    return np.exp(edges)


def assign_fine_bin(period: np.ndarray, edges: np.ndarray) -> np.ndarray:
    """0-based bin index per flow from its inherited channel period.

    -1 marks a flow whose period is NaN or falls outside [edges[0], edges[-1]]
    -- excluded rather than clipped into an end bin, so a bin's population
    never silently absorbs out-of-range channels.
    """
    period = np.asarray(period, dtype=float)
    idx = np.digitize(period, edges[1:-1], right=False)
    out_of_range = (period < edges[0]) | (period > edges[-1])
    idx = np.where(np.isfinite(period) & ~out_of_range, idx, -1)
    return idx


def bin_centers(edges: np.ndarray) -> np.ndarray:
    """Geometric mean of each bin's edges -- the right centre for a log axis."""
    return np.sqrt(edges[:-1] * edges[1:])


def per_finebin_metrics(pred: dict, edges: np.ndarray) -> dict:
    """Per-fine-bin metrics, aggregated over seeds -- evaluate.per_band's
    logic applied to fine bins instead of B1/B2/B3, plus the per-seed DR
    series each bin needs for crossing_point_per_seed.

    ``pred`` must carry "period" -- pass periods= to models.cross_validate.
    """
    if "period" not in pred:
        raise ValueError(
            "prediction dict has no 'period'; call cross_validate(..., periods=...)")
    bins = assign_fine_bin(pred["period"], edges)
    seeds = np.unique(pred["seed"])
    out: dict[int, dict] = {}
    for bi in range(len(edges) - 1):
        mask_b = bins == bi
        if not mask_b.any():
            continue
        per_seed = []
        for s in seeds:
            m = mask_b & (pred["seed"] == s)
            if m.sum() == 0:
                continue
            per_seed.append(metrics(pred["y_true"][m], pred["y_score"][m]))
        if not per_seed:
            continue
        agg: dict = {}
        for k in ("DR", "FAR", "P", "R", "F1"):
            vals = np.array([d[k] for d in per_seed], dtype=float)
            agg[k] = float(np.nanmean(vals))
            agg[k + "_sd"] = float(np.nanstd(vals))
            agg[k + "_per_seed"] = vals.tolist()
        agg["n"] = int(mask_b.sum() // max(len(seeds), 1))
        agg["n_pos"] = int(per_seed[0]["n_pos"])
        out[bi] = agg
    return out


# --------------------------------------------------------------------------
# Crossing-point estimation
# --------------------------------------------------------------------------
def find_crossings(x: np.ndarray, y: np.ndarray, level: float) -> list[float]:
    """Every x where the piecewise-linear interpolant of (x, y) crosses level.

    Detection-rate-vs-interval need not be monotonic -- it isn't for k-NN in
    this study -- so this returns every crossing rather than assuming one.
    Bins with no data are simply absent from x/y; the interpolant never
    bridges a genuinely missing bin's gap silently because the caller builds
    x/y only from bins that exist (see crossing_point_per_seed).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if x.size < 2:
        return []
    order = np.argsort(x)
    x, y = x[order], y[order]
    out: list[float] = []
    for i in range(len(x) - 1):
        y0, y1 = y[i] - level, y[i + 1] - level
        if y0 == 0:
            out.append(float(x[i]))
        elif y0 * y1 < 0:
            frac = y0 / (y0 - y1)
            out.append(float(x[i] + frac * (x[i + 1] - x[i])))
    if y[-1] == level:
        out.append(float(x[-1]))
    return sorted(set(round(v, 6) for v in out))


def crossing_point_per_seed(bin_metrics: dict, centers: np.ndarray,
                            tau: float = C.TAU) -> dict:
    """The smallest-period crossing of tau, estimated independently per seed.

    The spread across seeds stands in for a confidence interval. With five
    seeds this is a range, not a proper CI, and is reported as a range, not
    dressed up as one. A detector that never crosses tau (Random Forest, in
    this study) has n_found=0 for a reason -- that is itself the finding, not
    a gap to paper over with an extrapolated number.
    """
    bin_ids = sorted(bin_metrics.keys())
    if not bin_ids:
        return {"first_crossing_per_seed": [], "mean": None, "range": None,
                "n_seeds": 0, "n_found": 0}
    n_seeds = len(bin_metrics[bin_ids[0]]["DR_per_seed"])
    firsts: list[float | None] = []
    for si in range(n_seeds):
        xs = np.array([centers[bi] for bi in bin_ids])
        ys = np.array([bin_metrics[bi]["DR_per_seed"][si] for bi in bin_ids])
        crossings = find_crossings(xs, ys, tau)
        firsts.append(crossings[0] if crossings else None)
    found = [f for f in firsts if f is not None]
    return {
        "first_crossing_per_seed": firsts,
        "mean": float(np.mean(found)) if found else None,
        "range": [float(min(found)), float(max(found))] if found else None,
        "n_seeds": n_seeds,
        "n_found": len(found),
    }


# --------------------------------------------------------------------------
# Top-level assembly
# --------------------------------------------------------------------------
def dose_response(predictions: dict, channels: "pd.DataFrame",  # noqa: F821
                  n_bins: int = 8, tau: float = C.TAU) -> dict:
    """Full dose-response result: per-detector fine-bin metrics + crossing point.

    ``predictions`` is {model: pred} where each pred came from
    models.cross_validate(..., periods=periods). ``channels`` is the
    per-channel table from intervals.derive() (its "period" column sets the
    bin edges).
    """
    edges = fine_bin_edges(channels["period"].to_numpy(), n_bins=n_bins)
    centers = bin_centers(edges)
    out: dict = {
        "edges": edges.tolist(),
        "centers": centers.tolist(),
        "tau": tau,
        "models": {},
    }
    for model, pred in predictions.items():
        bm = per_finebin_metrics(pred, edges)
        crossing = crossing_point_per_seed(bm, centers, tau=tau)
        out["models"][model] = {
            "bins": {int(k): v for k, v in bm.items()},
            "crossing": crossing,
        }
    return out


def cross_corpus_report(cc_predictions: dict) -> dict:
    """Per-band metrics for each direction of models.cross_corpus_eval's output.

    Reuses evaluate.per_band directly -- cc_predictions[direction][model]
    already carries "band" and "seed" in the same shape per_band expects,
    since models.cross_corpus_eval was called with bands= to produce it.
    """
    out: dict = {}
    for direction, per_model in cc_predictions.items():
        out[direction] = {m: per_band(pred) for m, pred in per_model.items()}
    return out


def write_dose_response_macros(dr: dict, path: Path) -> None:
    """\\newcommand macros for the crossing-point numbers, mirroring
    evaluate.write_macros so the extension's prose can cite them the same way
    the main paper cites \\ResDRrfBone etc."""
    def cmd(name, value):
        return r"\newcommand{\%s}{%s}" % (name, value)

    mdl = {"Suricata": "sur", "Random Forest": "rf", "Gradient Boosting": "gb", "k-NN": "knn"}
    out = [r"% Generated by src/generalize.py -- regenerate, do not hand-edit."]
    out.append(cmd("DRNBins", str(len(dr["centers"]))))
    for m, key in mdl.items():
        if m not in dr["models"]:
            continue
        c = dr["models"][m]["crossing"]
        out.append(cmd(f"DRXFound{key}", str(c["n_found"])))
        if c["mean"] is not None:
            out.append(cmd(f"DRXMean{key}", f"{c['mean']:.0f}"))
            out.append(cmd(f"DRXLo{key}", f"{c['range'][0]:.0f}"))
            out.append(cmd(f"DRXHi{key}", f"{c['range'][1]:.0f}"))
        else:
            out.append(cmd(f"DRXMean{key}", "none"))
    Path(path).write_text("\n".join(out) + "\n", encoding="utf-8")
    log.info("wrote %s", path)


__all__ = [
    "fine_bin_edges", "assign_fine_bin", "bin_centers", "per_finebin_metrics",
    "find_crossings", "crossing_point_per_seed", "dose_response",
    "cross_corpus_report", "write_dose_response_macros",
]

In [ ]:
%%writefile src/suricata.py
"""Stage E (signature branch) -- Suricata baseline.

Suricata is run offline over the corpus packet captures with the Emerging
Threats Open ruleset in its DEFAULT configuration. The ruleset is deliberately
not tuned to the traffic: tuning a rule to the captures it is then evaluated on
would make the comparison circular, and the operationally interesting question
is what a defender running a standard configuration actually detects.
"""
from __future__ import annotations

import json
import logging
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from . import config as C

log = logging.getLogger(__name__)


def suricata_available() -> bool:
    return shutil.which(C.SURICATA_BIN) is not None


def run_suricata(pcap: Path, out_dir: Path,
                 rules: Path = C.SURICATA_RULES) -> Path:
    """Replay a pcap through Suricata; return the path to eve.json."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if not Path(rules).exists():
        raise FileNotFoundError(
            f"ruleset not found at {rules}. Fetch ET Open with:\n"
            "  make rules")
    cmd = [C.SURICATA_BIN, "-r", str(pcap), "-S", str(rules),
           "-l", str(out_dir), "--set", "outputs.1.eve-log.enabled=yes"]
    log.info("suricata: %s", " ".join(cmd))
    subprocess.run(cmd, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    eve = out_dir / "eve.json"
    if not eve.exists():
        raise FileNotFoundError(f"suricata produced no eve.json in {out_dir}")
    return eve


def parse_alerts(eve_json: Path) -> pd.DataFrame:
    """Extract alert five-tuples and timestamps from eve.json."""
    recs = []
    with open(eve_json, "rt") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                ev = json.loads(line)
            except json.JSONDecodeError:
                continue
            if ev.get("event_type") != "alert":
                continue
            recs.append({
                "ts": pd.Timestamp(ev["timestamp"]).timestamp(),
                "src_ip": ev.get("src_ip"),
                "dest_ip": ev.get("dest_ip"),
                "dest_port": ev.get("dest_port"),
                "proto": str(ev.get("proto", "")).lower(),
                "signature": ev.get("alert", {}).get("signature", ""),
            })
    df = pd.DataFrame.from_records(recs)
    log.info("parsed %d alerts from %s", len(df), eve_json.name)
    return df


def attribute_alerts(flows: pd.DataFrame, alerts: pd.DataFrame,
                     tol: float = C.ALERT_MATCH_TOLERANCE_S) -> np.ndarray:
    """Map alerts onto flows by five-tuple and timestamp proximity.

    Returns a binary score array aligned with ``flows``. Suricata emits a
    decision, not a probability, so its ROC contribution is a single operating
    point rather than a curve -- the figures handle it that way rather than
    fabricating a curve.
    """
    score = np.zeros(len(flows), dtype=float)
    if alerts is None or alerts.empty:
        log.warning("no alerts to attribute; Suricata scores all-zero")
        return score

    key_cols = ["id.orig_h", "id.resp_h", "id.resp_p", "proto"]
    fl = flows[key_cols + ["ts"]].copy()
    fl["proto"] = fl["proto"].astype(str).str.lower()
    fl["id.resp_p"] = pd.to_numeric(fl["id.resp_p"], errors="coerce")

    al = alerts.copy()
    al["dest_port"] = pd.to_numeric(al["dest_port"], errors="coerce")

    index: dict[tuple, list[float]] = {}
    for r in al.itertuples(index=False):
        index.setdefault((r.src_ip, r.dest_ip, r.dest_port, r.proto), []).append(r.ts)
    for k in index:
        index[k].sort()

    for i, r in enumerate(fl.itertuples(index=False)):
        k = (r[0], r[1], r[2], r[3])
        times = index.get(k)
        if not times:
            continue
        pos = np.searchsorted(times, r.ts)
        for j in (pos - 1, pos):
            if 0 <= j < len(times) and abs(times[j] - r.ts) <= tol:
                score[i] = 1.0
                break

    log.info("attributed alerts to %d/%d flows", int(score.sum()), len(score))
    return score


def synthetic_alerts(flows: pd.DataFrame, seed: int = 11) -> np.ndarray:
    """Stand-in used only by the smoke test, when no pcap or Suricata is present.

    Recall degrades with band, which is the behaviour a rule engine keyed to
    short-lived indicators is expected to show. This is NOT a result and is
    never used by ``run_all.py --real``.
    """
    rng = np.random.default_rng(seed)
    recall = {"B1": 0.62, "B2": 0.41, "B3": 0.18}
    fpr = {"B1": 0.020, "B2": 0.018, "B3": 0.015}
    score = np.zeros(len(flows), dtype=float)
    y = flows["y"].to_numpy()
    bands = flows["band"].to_numpy()
    for b in C.BANDS:
        m = bands == b
        pos = np.flatnonzero(m & (y == 1))
        neg = np.flatnonzero(m & (y == 0))
        if pos.size:
            score[rng.choice(pos, int(recall[b] * pos.size), replace=False)] = 1.0
        if neg.size:
            score[rng.choice(neg, int(fpr[b] * neg.size), replace=False)] = 1.0
    return score

In [ ]:
%%writefile src/synth.py
"""Synthetic Zeek conn.log.labeled generator -- for the smoke test only.

The corpora are large (IoT-23 is 21 GB with pcaps). This module writes a small,
structurally faithful conn.log.labeled so the pipeline can be exercised end to
end in seconds, on a laptop, before anyone downloads anything.

Nothing produced here is a research result. ``run_all.py --real`` never touches
this module.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

HEADER = """#separator \\x09
#set_separator\t,
#empty_field\t(empty)
#unset_field\t-
#path\tconn
#fields\tts\tuid\tid.orig_h\tid.orig_p\tid.resp_h\tid.resp_p\tproto\tservice\tduration\torig_bytes\tresp_bytes\tconn_state\torig_pkts\tresp_pkts\tlabel
#types\ttime\tstring\taddr\tport\taddr\tport\tenum\tstring\tinterval\tcount\tcount\tstring\tcount\tcount\tstring
"""


def _uid(rng) -> str:
    return "C" + "".join(rng.choice(list("abcdefghijklmnopqrstuvwxyz0123456789"), 12))


def write_synthetic_conn_log(path: Path, seed: int = 20260829,
                             n_c2_channels: int = 90,
                             n_benign_channels: int = 220) -> Path:
    """Write a synthetic labelled conn.log covering all three interval bands."""
    rng = np.random.default_rng(seed)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows: list[str] = []

    # --- C2 channels: genuinely periodic, periods spanning the three bands ---
    period_pool = np.concatenate([
        rng.lognormal(np.log(25), 0.35, n_c2_channels // 3),      # B1
        rng.lognormal(np.log(240), 0.30, n_c2_channels // 3),     # B2
        rng.lognormal(np.log(2400), 0.40, n_c2_channels - 2 * (n_c2_channels // 3)),
    ])
    for ci, period in enumerate(period_pool):
        src = f"192.168.1.{10 + ci % 200}"
        dst = f"203.0.{113 + ci % 3}.{20 + ci % 200}"
        dport = int(rng.choice([443, 8080, 80, 8443]))
        n = int(rng.integers(14, 60))
        jitter = float(rng.uniform(0.03, 0.30))
        t = float(rng.uniform(1.6e9, 1.6e9 + 1e5))
        # longer periods carry weaker volume signal -- the effect under study
        vol = max(0.25, 1.6 - 0.28 * np.log10(max(period, 1.0)))
        for _ in range(n):
            dur = abs(rng.normal(0.55, 0.18))
            ob = max(40, int(rng.normal(260 * vol, 55)))
            rb = max(40, int(rng.normal(340 * vol, 70)))
            rows.append("\t".join([
                f"{t:.6f}", _uid(rng), src, str(int(rng.integers(1024, 65535))),
                dst, str(dport), "tcp", "ssl", f"{dur:.6f}", str(ob), str(rb),
                "SF", str(int(rng.integers(4, 12))), str(int(rng.integers(4, 12))),
                "C&C-HeartBeat",
            ]))
            t += max(0.5, period * (1.0 + rng.normal(0, jitter)))

    # --- benign channels -------------------------------------------------
    # Two populations, deliberately. Bursty human traffic is easy to separate;
    # PERIODIC benign traffic (NTP polling, telemetry heartbeats, scheduled
    # backups) is the hard negative that makes this task non-trivial, and it is
    # what puts benign flows into every band so the false-alarm rate is defined
    # in B3 as well as B1.
    n_bursty = int(n_benign_channels * 0.6)
    n_periodic = n_benign_channels - n_bursty

    for ci in range(n_bursty):
        src = f"192.168.1.{10 + ci % 200}"
        dst = f"198.51.100.{5 + ci % 240}"
        dport = int(rng.choice([443, 80, 53, 22, 8080]))
        n = int(rng.integers(12, 70))
        t = float(rng.uniform(1.6e9, 1.6e9 + 1e5))
        for _ in range(n):
            rows.append("\t".join([
                f"{t:.6f}", _uid(rng), src, str(int(rng.integers(1024, 65535))),
                dst, str(dport), str(rng.choice(["tcp", "udp"])),
                str(rng.choice(["http", "ssl", "dns", "-"])),
                f"{abs(rng.normal(3.2, 2.6)):.6f}",
                str(max(60, int(rng.lognormal(7.2, 1.1)))),
                str(max(60, int(rng.lognormal(7.8, 1.3)))),
                str(rng.choice(["SF", "S0", "RSTO"])),
                str(int(rng.integers(2, 40))), str(int(rng.integers(2, 40))),
                "Benign",
            ]))
            t += float(rng.exponential(45.0)) + 0.5

    benign_periods = np.concatenate([
        rng.lognormal(np.log(32), 0.35, n_periodic // 3),      # B1: NTP-like
        rng.lognormal(np.log(300), 0.30, n_periodic // 3),     # B2: telemetry
        rng.lognormal(np.log(3000), 0.45,
                      n_periodic - 2 * (n_periodic // 3)),     # B3: backups
    ])
    for ci, period in enumerate(benign_periods):
        src = f"192.168.2.{10 + ci % 200}"
        dst = f"198.51.101.{5 + ci % 240}"
        dport = int(rng.choice([123, 443, 8883, 514]))
        n = int(rng.integers(14, 55))
        jitter = float(rng.uniform(0.05, 0.40))
        t = float(rng.uniform(1.6e9, 1.6e9 + 1e5))
        for _ in range(n):
            rows.append("\t".join([
                f"{t:.6f}", _uid(rng), src, str(int(rng.integers(1024, 65535))),
                dst, str(dport), "tcp", str(rng.choice(["ssl", "-"])),
                f"{abs(rng.normal(0.9, 0.5)):.6f}",
                str(max(50, int(rng.normal(420, 160)))),
                str(max(50, int(rng.normal(520, 190)))),
                "SF", str(int(rng.integers(3, 16))), str(int(rng.integers(3, 16))),
                "Benign",
            ]))
            t += max(0.5, period * (1.0 + rng.normal(0, jitter)))

    # --- background traffic: must be EXCLUDED by preprocess.assign_class ---
    for ci in range(30):
        src = f"10.0.0.{ci + 2}"
        for _ in range(int(rng.integers(10, 25))):
            rows.append("\t".join([
                f"{rng.uniform(1.6e9, 1.6e9 + 1e5):.6f}", _uid(rng), src,
                str(int(rng.integers(1024, 65535))), "192.0.2.9", "443", "tcp",
                "-", "1.0", "100", "100", "SF", "3", "3", "Background-Unknown",
            ]))

    rng.shuffle(rows)
    path.write_text(HEADER + "\n".join(rows) + "\n#close\t2026-08-29-00-00-00\n",
                    encoding="utf-8")
    return path

In [ ]:
%%writefile src/figures.py
"""Figure generation. Every plot the paper needs that a diagram tool cannot draw.

All figures are grayscale with hatching or line style as a second encoding, so
identity never rests on colour alone and the figures survive black-and-white
printing. Sizes follow IEEE geometry: 3.5 in single column, 7.16 in double.
"""
from __future__ import annotations

import logging
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import (auc, average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_curve)

from . import config as C

log = logging.getLogger(__name__)

COL_W, DBL_W = 3.5, 7.16
GREY = {"Suricata": "0.78", "Random Forest": "0.48", "Gradient Boosting": "0.30", "k-NN": "0.16"}
HATCH = {"Suricata": "//", "Random Forest": "", "Gradient Boosting": "xx", "k-NN": ".."}
BAND_LS = {"B1": "-", "B2": "--", "B3": ":"}
BAND_AX = {"B1": "B1\n(<60 s)", "B2": "B2\n(60–600 s)", "B3": "B3\n(≥600 s)"}
BAND_LONG = {"B1": "B1 sub-minute", "B2": "B2 minutes", "B3": "B3 ≥10 min"}


def apply_style() -> None:
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Nimbus Roman", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
        "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
        "axes.linewidth": 0.7, "grid.linewidth": 0.4, "lines.linewidth": 1.2,
        "xtick.major.width": 0.7, "ytick.major.width": 0.7,
        "figure.dpi": 150, "savefig.dpi": 300,
        "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42, "ps.fonttype": 42,
    })


def _tidy(ax, grid_axis="y"):
    ax.spines[["top", "right"]].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color="0.85", lw=0.4, zorder=0)
        ax.set_axisbelow(True)


# Set by make_all(). When true, every figure carries a diagonal SYNTHETIC DATA
# stamp. A figure produced from the smoke test must never be mistakable for a
# result -- it is the single easiest way to end up submitting a fabricated plot.
WATERMARK = False


WATERMARK_TEXT = "SYNTHETIC DATA"


def _stamp(fig):
    """Add the synthetic-data watermark. Returns the text artist."""
    t = fig.text(0.5, 0.5, WATERMARK_TEXT, fontsize=34, color="red",
                 alpha=0.22, ha="center", va="center", rotation=30,
                 zorder=1000, transform=fig.transFigure)
    # Excluded from the tight bounding box: otherwise the stamp inflates every
    # saved figure and the paper silently grows a page.
    t.set_in_layout(False)
    return t


def _save(fig, name, out: Path):
    out.mkdir(parents=True, exist_ok=True)
    if WATERMARK:
        _stamp(fig)
    for ext in ("pdf", "png"):
        fig.savefig(out / f"{name}.{ext}")
    plt.close(fig)
    log.info("wrote %s.pdf/.png%s", name, "  [SYNTHETIC watermark]" if WATERMARK else "")


# --------------------------------------------------------------------------
def fig_detection_far(per_band: dict, out: Path) -> None:
    """Headline figure. Two panels -- never a twin axis; DR and FAR are
    different measures and must not share a scale."""
    models = [m for m in C.DETECTORS if m in per_band]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(DBL_W, 2.5))
    x = np.arange(len(C.BANDS)); w = 0.26

    for i, m in enumerate(models):
        off = (i - (len(models) - 1) / 2) * w
        dr = [per_band[m].get(b, {}).get("DR", np.nan) for b in C.BANDS]
        drsd = [per_band[m].get(b, {}).get("DR_sd", 0.0) for b in C.BANDS]
        far = [per_band[m].get(b, {}).get("FAR", np.nan) for b in C.BANDS]
        ax1.bar(x + off, dr, w * 0.92, yerr=drsd, capsize=2, label=m,
                color=GREY[m], edgecolor="black", linewidth=0.7,
                hatch=HATCH[m], zorder=3, error_kw={"lw": 0.7})
        ax2.bar(x + off, far, w * 0.92, label=m, color=GREY[m],
                edgecolor="black", linewidth=0.7, hatch=HATCH[m], zorder=3)

    ax1.axhline(C.TAU, ls=(0, (4, 3)), lw=0.9, color="black", zorder=4)
    ax1.text(x[0] - 0.42, C.TAU + 0.025, r"$\tau=%.2f$" % C.TAU,
             ha="left", va="bottom", fontsize=7.5)

    ax1.set_ylabel("Detection rate"); ax1.set_ylim(0, 1.02)
    ax1.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    ax2.set_ylabel("False-alarm rate")
    ax2.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=1))
    for ax, t in ((ax1, "(a) Detection rate by band"),
                  (ax2, "(b) False-alarm rate by band")):
        ax.set_xticks(x, [BAND_AX[b] for b in C.BANDS])
        ax.set_xlabel("Beacon-interval band"); ax.set_title(t); _tidy(ax)
    ax1.legend(frameon=False, loc="upper center", ncol=3, handlelength=1.3,
               borderpad=0.2, columnspacing=1.1, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(); _save(fig, "fig_detection_far", out)


def fig_roc(preds: dict, sur_points: dict | None, out: Path) -> None:
    models = [m for m in ("Random Forest", "Gradient Boosting", "k-NN") if m in preds]
    fig, axes = plt.subplots(1, len(models), figsize=(DBL_W, 2.7), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, models):
        p = preds[m]
        for b in C.BANDS:
            msk = p["band"] == b
            if msk.sum() < 10 or len(np.unique(p["y_true"][msk])) < 2:
                continue
            fpr, tpr, _ = roc_curve(p["y_true"][msk], p["y_score"][msk])
            ax.plot(fpr, tpr, BAND_LS[b], color="black", lw=1.2, zorder=3,
                    label=f"{BAND_LONG[b]} (AUC {auc(fpr, tpr):.3f})")
        if sur_points:
            for i, b in enumerate(C.BANDS):
                if b not in sur_points:
                    continue
                ax.plot(sur_points[b]["FAR"], sur_points[b]["DR"], "o", ms=4.5,
                        mfc="white", mec="black", mew=0.9, ls="none", zorder=4,
                        label="Suricata operating point" if i == 0 else None)
        ax.plot([0, 1], [0, 1], color="0.75", lw=0.7, zorder=1)
        ax.set_xlabel("False positive rate"); ax.set_title(m)
        ax.set_xlim(-.02, 1.02); ax.set_ylim(-.02, 1.02)
        _tidy(ax, "both")
        ax.legend(frameon=False, loc="lower right", handlelength=1.9, borderpad=0.2)
    axes[0].set_ylabel("True positive rate")
    fig.tight_layout(); _save(fig, "fig_roc", out)


def fig_pr(preds: dict, out: Path) -> None:
    models = [m for m in ("Random Forest", "Gradient Boosting", "k-NN") if m in preds]
    fig, axes = plt.subplots(1, len(models), figsize=(DBL_W, 2.7), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, models):
        p = preds[m]
        for b in C.BANDS:
            msk = p["band"] == b
            if msk.sum() < 10 or len(np.unique(p["y_true"][msk])) < 2:
                continue
            y, s = p["y_true"][msk], p["y_score"][msk]
            pr, rc, _ = precision_recall_curve(y, s)
            ax.plot(rc, pr, BAND_LS[b], color="black", lw=1.2, zorder=3,
                    label=f"{BAND_LONG[b]} (AP {average_precision_score(y, s):.3f})")
            ax.axhline(y.mean(), color="0.85", lw=0.6, ls=(0, (1, 2)), zorder=1)
        ax.set_xlabel("Recall"); ax.set_title(m)
        ax.set_xlim(-.02, 1.02); ax.set_ylim(0, 1.02); _tidy(ax, "both")
        ax.legend(frameon=False, loc="lower left", handlelength=1.9, borderpad=0.2)
    axes[0].set_ylabel("Precision")
    fig.tight_layout(); _save(fig, "fig_pr", out)


def fig_confusion(preds: dict, out: Path, model: str = "Random Forest") -> None:
    if model not in preds:
        return
    p = preds[model]
    fig, axes = plt.subplots(1, len(C.BANDS), figsize=(DBL_W, 2.25))
    for ax, b in zip(np.atleast_1d(axes), C.BANDS):
        msk = p["band"] == b
        if msk.sum() == 0:
            ax.axis("off"); continue
        cm = confusion_matrix(p["y_true"][msk],
                              (p["y_score"][msk] >= C.DECISION_THRESHOLD).astype(int),
                              labels=[0, 1], normalize="true")
        ax.imshow(cm, cmap="Greys", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                dark = cm[i, j] > 0.55
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                        fontsize=8.5, color="white" if dark else "black",
                        path_effects=[pe.withStroke(linewidth=1.8,
                                      foreground="black" if dark else "white")])
        ax.set_title(BAND_LONG[b])
        ax.set_xticks([0, 1], ["Benign", "C2"])
        ax.set_yticks([0, 1], ["Benign", "C2"])
        ax.set_xlabel("Predicted")
        for s in ax.spines.values():
            s.set_linewidth(0.7)
    np.atleast_1d(axes)[0].set_ylabel("Actual")
    fig.suptitle(f"{model} — row-normalised confusion matrices", y=1.04, fontsize=8.5)
    fig.tight_layout(); _save(fig, "fig_confusion", out)


def fig_interval_hist(periods: np.ndarray, out: Path) -> None:
    periods = np.asarray(periods, dtype=float)
    periods = periods[np.isfinite(periods) & (periods > 0)]
    if periods.size == 0:
        return
    fig, ax = plt.subplots(figsize=(COL_W, 2.3))
    bins = np.logspace(np.log10(periods.min()), np.log10(periods.max()), 42)
    ax.hist(periods, bins=bins, color="0.62", edgecolor="black", lw=0.5, zorder=3)
    ax.set_xscale("log")
    for cut in C.BAND_EDGES_S:
        ax.axvline(cut, ls=(0, (4, 3)), lw=0.9, color="black", zorder=4)
    top = ax.get_ylim()[1]
    lo, hi = C.BAND_EDGES_S
    for xpos, lab in ((np.sqrt(periods.min() * lo), "B1"),
                      (np.sqrt(lo * hi), "B2"),
                      (np.sqrt(hi * periods.max()), "B3")):
        ax.text(xpos, top * 0.93, lab, ha="center", va="top",
                fontsize=8, weight="bold")
    ax.set_xlabel("Derived beacon period (s, log scale)")
    ax.set_ylabel("Channels"); _tidy(ax)
    fig.tight_layout(); _save(fig, "fig_interval_hist", out)


def fig_importance(importance: dict, out: Path, top: int = 12) -> None:
    if not importance:
        return
    items = sorted(importance.items(), key=lambda kv: kv[1][0])[-top:]
    names = [k.replace("num__", "").replace("cat__", "") for k, _ in items]
    mean = np.array([v[0] for _, v in items])
    std = np.array([v[1] for _, v in items])
    derived = set(C.TIMING_FEATURES)
    fig, ax = plt.subplots(figsize=(COL_W, 2.9))
    y = np.arange(len(names))
    for yi, m, s, n in zip(y, mean, std, names):
        is_d = any(d in n for d in derived)
        ax.barh(yi, m, xerr=s, height=0.72, color="0.30" if is_d else "0.72",
                edgecolor="black", linewidth=0.6, hatch="" if is_d else "///",
                error_kw={"lw": 0.7, "capsize": 2}, zorder=3)
    ax.set_yticks(y, names)
    ax.set_xlabel("Permutation importance (mean F1 decrease)")
    _tidy(ax, "x")
    handles = [plt.Rectangle((0, 0), 1, 1, fc="0.30", ec="black", lw=0.6),
               plt.Rectangle((0, 0), 1, 1, fc="0.72", ec="black", lw=0.6, hatch="///")]
    ax.legend(handles, ["Derived timing features", "Flow-record features"],
              frameon=False, loc="lower right", handlelength=1.4, borderpad=0.2)
    fig.tight_layout(); _save(fig, "fig_importance", out)


def fig_k_sweep(k_sweep, out: Path) -> None:
    if k_sweep is None:
        return
    ks, mean, std = k_sweep
    fig, ax = plt.subplots(figsize=(COL_W, 2.1))
    ax.plot(ks, mean, "-o", color="black", ms=3.5, lw=1.1, zorder=3)
    ax.fill_between(ks, mean - std, mean + std, color="0.85", zorder=2)
    best = int(np.argmax(mean))
    ax.plot(ks[best], mean[best], "o", ms=6, mfc="white", mec="black", mew=1.1, zorder=4)
    ax.annotate(f"k = {ks[best]}", (ks[best], mean[best]), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=7.5)
    ax.set_xlabel("k (number of neighbours)"); ax.set_ylabel("Cross-validated F1")
    ax.set_xticks(ks); _tidy(ax)
    fig.tight_layout(); _save(fig, "fig_k_sweep", out)


def fig_rf_oob(rf_oob, out: Path) -> None:
    if rf_oob is None:
        return
    n, err = rf_oob
    fig, ax = plt.subplots(figsize=(COL_W, 2.1))
    ax.plot(n, err, "-", color="black", lw=1.1, zorder=3)
    sel = C.RF_PARAMS["n_estimators"]
    ax.axvline(sel, ls=(0, (4, 3)), lw=0.9, color="0.45", zorder=2)
    ax.text(sel * 1.02, ax.get_ylim()[1] * 0.92, f"n = {sel}\n(selected)",
            fontsize=7.5, va="top")
    ax.set_xlabel("Number of trees"); ax.set_ylabel("Out-of-bag error"); _tidy(ax)
    fig.tight_layout(); _save(fig, "fig_rf_oob", out)


DET_LS = {"Suricata": (0, (1, 1)), "Random Forest": "-", "Gradient Boosting": (0, (3, 1, 1, 1)), "k-NN": (0, (5, 2))}
DET_MARKER = {"Suricata": "s", "Random Forest": "o", "Gradient Boosting": "D", "k-NN": "^"}


def fig_shap_band(shap_band: dict, out: Path, top: int = 10) -> None:
    """Mean |SHAP| per feature, grouped bars side by side across bands.

    Same visual language as fig_importance (grayscale, hatch as the second
    encoding) but computed within each band separately, so a feature whose
    attribution shifts across B1/B2/B3 is visible as a bar that grows or
    shrinks across the three series rather than a single pooled number.
    """
    if not shap_band or not shap_band.get("bands"):
        return
    feats = [f.replace("num__", "").replace("cat__", "") for f in shap_band["features"]]
    bands_present = [b for b in C.BANDS if b in shap_band["bands"]]
    if not bands_present:
        return
    mat = np.array([shap_band["bands"][b] for b in bands_present])  # (n_bands, n_features)
    order = np.argsort(mat.mean(axis=0))[-top:]
    names = [feats[i] for i in order]
    mat = mat[:, order]

    fig, ax = plt.subplots(figsize=(DBL_W, 3.0))
    y = np.arange(len(names))
    w = 0.8 / len(bands_present)
    greys = {"B1": "0.75", "B2": "0.45", "B3": "0.15"}
    hatches = {"B1": "", "B2": "//", "B3": "xx"}
    for i, b in enumerate(bands_present):
        off = (i - (len(bands_present) - 1) / 2) * w
        ax.barh(y + off, mat[i], height=w * 0.92, label=b,
               color=greys.get(b, "0.5"), edgecolor="black", linewidth=0.6,
               hatch=hatches.get(b, ""), zorder=3)
    ax.set_yticks(y, names)
    ax.set_xlabel(f"Mean |SHAP| -- {shap_band.get('model', '')}")
    ax.legend(frameon=False, loc="lower right", handlelength=1.4, borderpad=0.2)
    _tidy(ax, "x")
    fig.tight_layout(); _save(fig, "fig_shap_band", out)


def fig_shap_cross_corpus(shap_cross: dict, out: Path, top: int = 10) -> None:
    """Mean |SHAP|, native corpus vs. transfer corpus, for the same fitted model.

    A feature whose bar changes height between the two series is a candidate
    mechanism for the cross-corpus DR collapse reported by
    models.cross_corpus_eval / fig_cross_corpus -- this figure is the
    "why", that one is the "that".
    """
    if not shap_cross:
        return
    feats = [f.replace("num__", "").replace("cat__", "") for f in shap_cross["features"]]
    native = np.array(shap_cross["native"])
    transfer = np.array(shap_cross["transfer"])
    order = np.argsort(np.maximum(native, transfer))[-top:]
    names = [feats[i] for i in order]
    native, transfer = native[order], transfer[order]

    fig, ax = plt.subplots(figsize=(DBL_W, 3.0))
    y = np.arange(len(names))
    w = 0.34
    tr_c, te_c = shap_cross.get("train_corpus", "train"), shap_cross.get("test_corpus", "test")
    ax.barh(y - w / 2, native, height=w * 0.92, label=f"native ({tr_c})",
           color="0.30", edgecolor="black", linewidth=0.6, zorder=3)
    ax.barh(y + w / 2, transfer, height=w * 0.92, label=f"transfer ({te_c})",
           color="0.75", edgecolor="black", linewidth=0.6, hatch="//", zorder=3)
    ax.set_yticks(y, names)
    ax.set_xlabel(f"Mean |SHAP| -- {shap_cross.get('model', '')}")
    ax.legend(frameon=False, loc="lower right", handlelength=1.4, borderpad=0.2)
    _tidy(ax, "x")
    fig.tight_layout(); _save(fig, "fig_shap_cross_corpus", out)


def fig_dose_response(dr: dict, out: Path) -> None:
    """Detection rate vs. continuous beacon period, one curve per detector.

    Detectors are distinguished by line style and marker shape (all plotted
    in black), the same convention fig_roc uses for its per-band curves --
    grey fill alone would not survive black-and-white printing. The fixed
    B1/B2/B3 edges are drawn as reference lines so the reader can relate this
    finer-grained view back to the paper's pre-registered bands. Per-seed
    spread is shown as a shaded band, not error bars, since with up to 8 fine
    bins on a log-x axis, bars overlap too much to read.
    """
    if not dr or not dr.get("models"):
        return
    centers = np.asarray(dr["centers"], dtype=float)
    tau = dr["tau"]
    fig, ax = plt.subplots(figsize=(DBL_W, 2.6))
    for name in C.DETECTORS:
        if name not in dr["models"]:
            continue
        bins = dr["models"][name]["bins"]
        bin_ids = sorted(bins.keys(), key=int)
        if not bin_ids:
            continue
        x = centers[[int(b) for b in bin_ids]]
        mean = np.array([bins[b]["DR"] for b in bin_ids])
        sd = np.array([bins[b]["DR_sd"] for b in bin_ids])
        ax.plot(x, mean, ls=DET_LS[name], marker=DET_MARKER[name], ms=4.0,
                mfc="white", mec="black", color="black", lw=1.2,
                label=name, zorder=3)
        ax.fill_between(x, np.clip(mean - sd, 0, 1), np.clip(mean + sd, 0, 1),
                        color="0.6", alpha=0.15, zorder=2)
        cross = dr["models"][name]["crossing"]
        if cross["mean"] is not None:
            ax.axvline(cross["mean"], color="0.4", ls=DET_LS[name], lw=0.8, zorder=1)
    ax.axhline(tau, color="black", ls=(0, (4, 3)), lw=0.9, zorder=1)
    ax.text(centers[0], tau + 0.02, rf"$\tau$={tau:.2f}", fontsize=7.5, va="bottom")
    for cut in C.BAND_EDGES_S:
        ax.axvline(cut, color="0.82", ls="-", lw=0.6, zorder=0)
    ax.set_xscale("log")
    ax.set_xlabel("Beacon period (s, log scale)")
    ax.set_ylabel("Detection rate")
    ax.set_ylim(-0.03, 1.03)
    ax.legend(frameon=False, loc="lower left", ncol=3, handlelength=2.2)
    _tidy(ax)
    fig.tight_layout(); _save(fig, "fig_dose_response", out)


def fig_cross_corpus(cc: dict, out: Path) -> None:
    """DR by band for each cross-corpus direction, grouped per detector.

    Both cold train-on-one/test-on-the-other directions side by side, one
    panel per band, so a reader can see at a glance whether a detector's
    generalisation gap is direction-dependent. This does not include the
    pooled within-distribution DR from the main protocol -- the paper's main
    results table already reports that; this figure is only about the two
    cross-corpus directions relative to each other.
    """
    if not cc:
        return
    directions = [d for d in ("iot23_to_ctu13", "ctu13_to_iot23") if d in cc]
    if not directions:
        return
    fig, axes = plt.subplots(1, len(C.BANDS), figsize=(DBL_W, 2.4), sharey=True)
    if len(C.BANDS) == 1:
        axes = [axes]
    width = 0.8 / len(directions)
    for ax, band in zip(axes, C.BANDS):
        xpos = np.arange(len(C.DETECTORS))
        for di, direction in enumerate(directions):
            vals = []
            for m in C.DETECTORS:
                v = cc.get(direction, {}).get(m, {}).get(band)
                vals.append(v["DR"] if v else np.nan)
            ax.bar(xpos + di * width - width * (len(directions) - 1) / 2, vals,
                  width=width * 0.9, color="0.30" if di == 0 else "0.70",
                  edgecolor="black", lw=0.6,
                  hatch="" if di == 0 else "///",
                  label=direction.replace("_to_", "→") if ax is axes[0] else None,
                  zorder=3)
        ax.set_xticks(xpos, [m.replace(" ", "\n") for m in C.DETECTORS])
        ax.set_title(BAND_LONG[band], fontsize=8)
        _tidy(ax, "y")
    axes[0].set_ylabel("Detection rate")
    axes[0].set_ylim(0, 1.05)
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, frameon=False, loc="upper center",
                  ncol=len(directions), bbox_to_anchor=(0.5, 1.08))
    fig.tight_layout(); _save(fig, "fig_cross_corpus", out)


def make_all(preds: dict, per_band: dict, periods, importance,
             k_sweep, rf_oob, out: Path = C.FIGURES,
             watermark: bool = False) -> None:
    global WATERMARK
    WATERMARK = bool(watermark)
    apply_style()
    out = Path(out)
    sur = per_band.get("Suricata")
    fig_detection_far(per_band, out)
    fig_roc(preds, sur, out)
    fig_pr(preds, out)
    fig_confusion(preds, out)
    fig_interval_hist(periods, out)
    fig_importance(importance, out)
    fig_k_sweep(k_sweep, out)
    fig_rf_oob(rf_oob, out)

In [ ]:
%%writefile src/explain.py
"""Stage G -- SHAP explainability (exploratory, not pre-registered).

Two questions this answers that Stage E's permutation importance
(models.fit_final_and_importance) does not, because that is computed once,
pooled across bands and corpora:

1. PER-BAND. Does the model lean on different features as beacon interval
   grows? A feature-attribution shift across B1/B2/B3 is a candidate
   mechanism for the dose-response curve in generalize.py, not just another
   plot of the same DR numbers.
2. CROSS-CORPUS. Does the SAME fitted model's attribution change when it
   scores flows from a corpus it never trained on? A shift here is a
   candidate explanation for the collapse reported by
   models.cross_corpus_eval -- evidence of *why*, not only *that*.

Like generalize.py, both analyses sit outside the pre-registered tau/delta
breakdown criterion (Section III, fixed in config.py before any detector
ran) -- they did not exist until after the main results did, so they are
reported as exploratory, not confirmatory.
"""
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from . import config as C
from .features import build_transformer, feature_columns, feature_names
from .models import _assert_no_group_overlap, _make_model, _resample

log = logging.getLogger(__name__)

try:
    import shap
    HAVE_SHAP = True
except ImportError:                                     # pragma: no cover
    HAVE_SHAP = False
    log.warning("shap not installed: explainability stage will be skipped. "
               "Install it (pip install shap) to reproduce this analysis.")

# TreeExplainer is fast, but this keeps per-band / per-corpus runtime bounded
# on the largest bands (B1 has 60k+ rows).
MAX_SHAP_ROWS = 500


def _tree_shap(model, Xt: np.ndarray) -> np.ndarray:
    """SHAP values for the POSITIVE class, normalised to shape (n, n_features).

    Random Forest's TreeExplainer returns one array per class; Gradient
    Boosting's (HistGradientBoostingClassifier) returns a single array
    already relative to the positive class. Both are normalised here so
    callers never special-case the model.
    """
    expl = shap.TreeExplainer(model)
    sv = np.asarray(expl.shap_values(Xt))
    if sv.ndim == 3:                       # (n, n_features, n_classes)
        sv = sv[:, :, 1]
    return sv


def _fit_final(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
              model_name: str, seed: int):
    """One grouped train/test split, fit on the training side only.

    Deliberately separate from models.fit_final_and_importance (which is
    Random Forest-only): this needs to fit whichever model_name is asked
    for, Gradient Boosting by default since it is the strongest in-corpus
    detector as of the run this stage was added for.
    """
    num, cat = feature_columns(X)
    gkf = GroupKFold(n_splits=C.N_FOLDS)
    tr, te = next(gkf.split(X, y, groups=groups))
    _assert_no_group_overlap(groups, tr, te)

    ct = build_transformer(num, cat)
    Xtr = ct.fit_transform(X.iloc[tr])
    Xte = ct.transform(X.iloc[te])
    Xtr_r, ytr_r = _resample(Xtr, y[tr], seed)

    model = _make_model(model_name, {}, seed).fit(Xtr_r, ytr_r)
    return model, ct, te, Xte


def explain_per_band(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                     bands: np.ndarray, model_name: str = "Gradient Boosting",
                     seed: int = C.RANDOM_SEEDS[0],
                     max_rows: int = MAX_SHAP_ROWS) -> dict | None:
    """Mean |SHAP| per feature, computed separately within each band.

    Returns None if shap is not installed -- callers must check for that
    and skip the stage, the same pattern generalize.py's callers use for a
    missing second corpus.
    """
    if not HAVE_SHAP:
        return None
    model, ct, te, Xte = _fit_final(X, y, groups, model_name, seed)
    names = feature_names(ct)
    bands_te = bands[te]

    out = {"model": model_name, "features": names, "bands": {}}
    rng = np.random.default_rng(seed)
    for b in C.BANDS:
        idx = np.flatnonzero(bands_te == b)
        if idx.size == 0:
            continue
        if idx.size > max_rows:
            idx = rng.choice(idx, size=max_rows, replace=False)
        sv = _tree_shap(model, Xte[idx])
        out["bands"][b] = np.abs(sv).mean(axis=0).tolist()
        log.info("SHAP per-band %s: %d rows sampled, %d features", b, idx.size, len(names))
    return out


def explain_cross_corpus(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                         corpus: np.ndarray, model_name: str = "Gradient Boosting",
                         seed: int = C.RANDOM_SEEDS[0],
                         max_rows: int = MAX_SHAP_ROWS,
                         train_corpus: str = "iot23") -> dict | None:
    """Mean |SHAP| for one fitted model, scored on native vs. transfer data.

    Trains on ``train_corpus`` in full (same protocol as
    models.cross_corpus_eval), then compares SHAP on a held-out grouped
    slice of the training corpus ("native") against the entire other corpus
    ("transfer"). A feature whose attribution shifts between the two is a
    candidate mechanism for the cross-corpus DR collapse -- not a repeat of
    the fact that it collapses.
    """
    if not HAVE_SHAP:
        return None
    corpora = set(np.unique(corpus))
    other = next((c for c in corpora if c != train_corpus), None)
    if other is None or train_corpus not in corpora:
        log.warning("explain_cross_corpus: need both corpora present; skipped")
        return None

    num, cat = feature_columns(X)
    tr_all = np.flatnonzero(corpus == train_corpus)
    te_other = np.flatnonzero(corpus == other)

    gkf = GroupKFold(n_splits=C.N_FOLDS)
    tr_inner, te_native = next(gkf.split(X.iloc[tr_all], y[tr_all], groups=groups[tr_all]))
    tr_idx = tr_all[tr_inner]
    native_idx = tr_all[te_native]

    ct = build_transformer(num, cat)
    Xtr = ct.fit_transform(X.iloc[tr_idx])
    Xtr_r, ytr_r = _resample(Xtr, y[tr_idx], seed)
    model = _make_model(model_name, {}, seed).fit(Xtr_r, ytr_r)
    names = feature_names(ct)
    rng = np.random.default_rng(seed)

    def _sample_shap(idx):
        if idx.size > max_rows:
            idx = rng.choice(idx, size=max_rows, replace=False)
        Xt = ct.transform(X.iloc[idx])
        return np.abs(_tree_shap(model, Xt)).mean(axis=0)

    native = _sample_shap(native_idx)
    transfer = _sample_shap(te_other)

    return {
        "model": model_name,
        "train_corpus": train_corpus,
        "test_corpus": other,
        "features": names,
        "native": native.tolist(),
        "transfer": transfer.tolist(),
    }


def write_explain_macros(per_band: dict | None, cross: dict | None, path: Path) -> None:
    """\\newcommand macros so the prose cites the top SHAP feature per band
    (and the top cross-corpus attribution shift) instead of hard-coding
    a name that could drift from the run that produced it."""
    def cmd(name, value):
        return r"\newcommand{\%s}{%s}" % (name, value)

    def clean(n: str) -> str:
        return n.replace("num__", "").replace("cat__", "").replace("_", r"\_")

    out = [r"% Generated by src/explain.py -- regenerate, do not hand-edit."]
    idx = {"B1": "Bone", "B2": "Btwo", "B3": "Bthree"}

    if per_band:
        feats = per_band["features"]
        for b, bkey in idx.items():
            if b not in per_band["bands"]:
                continue
            vals = per_band["bands"][b]
            top = int(np.argmax(vals))
            out.append(cmd(f"ShapTopFeat{bkey}", clean(feats[top])))
            out.append(cmd(f"ShapTopVal{bkey}", f"{vals[top]:.3f}"))

    if cross:
        feats = cross["features"]
        native = np.array(cross["native"])
        transfer = np.array(cross["transfer"])
        shift = np.abs(transfer - native)
        top = int(np.argmax(shift))
        out.append(cmd("ShapShiftFeat", clean(feats[top])))
        out.append(cmd("ShapShiftNative", f"{native[top]:.3f}"))
        out.append(cmd("ShapShiftTransfer", f"{transfer[top]:.3f}"))

    Path(path).write_text("\n".join(out) + "\n", encoding="utf-8")
    log.info("wrote %s", path)


In [ ]:
%%writefile run_all.py
#!/usr/bin/env python3
"""Quiet Signal -- end-to-end pipeline driver.

    python run_all.py --smoke     Synthetic conn.log, runs in ~1 min. Verifies the
                                  whole pipeline works before you download 21 GB.
    python run_all.py --real      The real corpora in data/. Produces every number
                                  and figure the paper reports.
    --extension                   Also runs the exploratory dose-response curve
                                  and cross-corpus generalisation check
                                  (src/generalize.py). Reuses the same
                                  out-of-fold predictions plus one extra
                                  training pass; not part of the pre-registered
                                  protocol. Needs both IoT-23 and CTU-13 under
                                  data/ for the cross-corpus half to run.
    --explain                     Also runs SHAP explainability (src/explain.py):
                                  per-band mean |SHAP| always; cross-corpus
                                  native-vs-transfer attribution shift if
                                  --extension is also given (needs both
                                  corpora). Requires `pip install shap`.
                                  Exploratory, not part of the pre-registered
                                  protocol.

Outputs
    results/results.json          all per-band metrics
    results/band_report.json      band populations, filter report
    paper/results_table.tex       the results table, best value bolded
    paper/results_macros.tex      \\newcommand macros the manuscript reads
    figures/*.pdf, *.png          all figures
    (with --extension)
    results/dose_response.json    fine-bin DR curve + crossing points per detector
    results/cross_corpus.json     per-band DR, both train/test corpus directions
    paper/dose_response_macros.tex  \\newcommand macros for the crossing points
    figures/fig_dose_response.*   DR vs. continuous period, one curve per detector
    figures/fig_cross_corpus.*    pooled vs. cross-corpus DR, grouped by band
    (with --explain)
    figures/fig_shap_band.*       mean |SHAP| per feature, grouped by band
    figures/fig_shap_cross_corpus.*  native vs. transfer attribution, per feature
    paper/explain_macros.tex      \\newcommand macros for the top SHAP features

The manuscript never hard-codes a number: it reads results_macros.tex. Re-run
this script and the paper updates itself.
"""
from __future__ import annotations

import argparse
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config as C
from src import evaluate, explain, features, figures, generalize, intervals, models, preprocess, suricata
from src.synth import write_synthetic_conn_log

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)-7s %(name)s: %(message)s",
                    datefmt="%H:%M:%S")
for _noisy in ("matplotlib", "fontTools", "PIL"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
log = logging.getLogger("run_all")

PAPER = C.ROOT / "paper"
PAPER.mkdir(exist_ok=True)


def discover_logs() -> list[Path]:
    logs = (sorted(C.DATA.rglob("conn.log.labeled"))
            + sorted(C.DATA.rglob("*.binetflow"))
            + sorted(C.DATA.rglob("*.binetflow.labeled"))
            + sorted(C.DATA.rglob("conn.log")))
    if not logs:
        sys.exit(
            "No connection records found under data/.\n"
            "  IoT-23 : per-scenario conn.log.labeled from\n"
            "           https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/\n"
            "           IndividualScenarios/<scenario>/bro/conn.log.labeled\n"
            "  CTU-13 : per-scenario labelled NetFlow from\n"
            "           https://mcfp.felk.cvut.cz/publicDatasets/\n"
            "           CTU-Malware-Capture-Botnet-<N>/\n"
            "           detailed-bidirectional-flow-labels/*.binetflow\n"
            "           (no Zeek and no pcaps needed)\n"
            "Or run:  python run_all.py --smoke")
    return logs


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--smoke", action="store_true",
                   help="synthetic data; verifies the pipeline end to end")
    g.add_argument("--real", action="store_true",
                   help="real corpora from data/")
    ap.add_argument("--seeds", type=int, default=len(C.RANDOM_SEEDS),
                    help="number of seeds to run (default: all)")
    ap.add_argument("--skip-diagnostics", action="store_true",
                    help="skip permutation importance, k-sweep and OOB curve. "
                         "The three main results figures are still produced; only "
                         "Figs. of importance/k-sweep/OOB are omitted. Useful on a "
                         "time-limited runtime.")
    ap.add_argument("--extension", action="store_true",
                    help="also run the exploratory dose-response and "
                         "cross-corpus generalisation analyses (src/generalize.py). "
                         "Not part of the pre-registered protocol -- see Section "
                         "'Future work' / the extension appendix. Adds one extra "
                         "training pass (cross-corpus) on top of the main run.")
    ap.add_argument("--finegrain-bins", type=int, default=8,
                    help="number of quantile-spaced period bins for the "
                         "dose-response curve (default: 8). Only used with "
                         "--extension.")
    ap.add_argument("--explain", action="store_true",
                    help="also run SHAP explainability (src/explain.py): "
                         "per-band always, cross-corpus native-vs-transfer "
                         "attribution shift if --extension is also given. "
                         "Exploratory, not part of the pre-registered protocol.")
    args = ap.parse_args()

    seeds = C.RANDOM_SEEDS[: max(1, args.seeds)]

    # ---------------------------------------------------------- Stage A + B
    if args.smoke:
        log.warning("SMOKE MODE -- synthetic data. Nothing here is a result.")
        p = write_synthetic_conn_log(C.DATA / "smoke" / "conn.log.labeled")
        logs = [p]
    else:
        logs = discover_logs()
        log.info("found %d connection log(s)", len(logs))

    flows, filt_rep = preprocess.prepare(logs)
    log.info("filter report: %s", filt_rep)

    # ---------------------------------------------------------- Stage C
    flows, channels = intervals.derive(flows)
    band_rep = intervals.band_report(channels)
    log.info("band report: %s", json.dumps(band_rep, indent=2, default=float))

    # The derived interval labels are the artefact released with the paper.
    channels.to_csv(C.RESULTS / "derived_channel_intervals.csv", index=False)

    # ---------------------------------------------------------- Stage D
    if args.extension:
        X, y, groups, bands, periods, corpus = features.assemble_ext(flows)
    else:
        X, y, groups, bands = features.assemble(flows)
        periods = corpus = None
    if len(np.unique(y)) < 2:
        sys.exit("only one class present after preprocessing -- check labels")

    # ---------------------------------------------------------- Stage E
    log.info("cross-validating with %d seed(s) x %d folds", len(seeds), C.N_FOLDS)
    preds = models.cross_validate(X, y, groups, bands, seeds=seeds,
                                  periods=periods, corpus=corpus)

    if args.real and suricata.suricata_available():
        log.info("Suricata found; replay must be run separately (see README)")
        eve = sorted(C.DATA.rglob("eve.json"))
        if eve:
            alerts = pd.concat([suricata.parse_alerts(e) for e in eve], ignore_index=True)
            sur_score = suricata.attribute_alerts(flows, alerts)
        else:
            log.warning("no eve.json under data/ -- Suricata baseline skipped")
            sur_score = None
    elif args.smoke:
        log.warning("smoke mode: using stand-in Suricata scores (NOT a result)")
        sur_score = suricata.synthetic_alerts(flows)
    else:
        log.warning("Suricata not available -- signature baseline skipped")
        sur_score = None

    sur_pred = None
    if sur_score is not None:
        # Suricata is deterministic: replicate across seeds so the aggregation
        # code path is identical for every detector.
        sur_pred = {
            "y_true": np.tile(y, len(seeds)),
            "y_score": np.tile(sur_score, len(seeds)),
            "band": np.tile(bands, len(seeds)),
            "seed": np.repeat(seeds, len(y)),
        }
        if periods is not None:
            sur_pred["period"] = np.tile(periods, len(seeds))

    # ---------------------------------------------------------- Stage F
    results = evaluate.evaluate_all(preds, sur_pred)
    evaluate.save_json(results, C.RESULTS / "results.json")
    evaluate.save_json({"bands": band_rep, "filter": filt_rep},
                       C.RESULTS / "band_report.json")
    evaluate.write_latex_table(results, PAPER / "results_table.tex")
    evaluate.write_macros(results, band_rep, filt_rep, PAPER / "results_macros.tex")

    # ---------------------------------------------------------- extension
    # Exploratory, outside the pre-registered protocol (Section III-I) --
    # neither analysis was fixed before this run's results existed.
    dr = cc = None
    if args.extension:
        log.info("dose-response curve: %d quantile-spaced bins", args.finegrain_bins)
        dr_preds = dict(preds)
        if sur_pred is not None and "period" in sur_pred:
            dr_preds["Suricata"] = sur_pred
        try:
            dr = generalize.dose_response(dr_preds, channels, n_bins=args.finegrain_bins)
            evaluate.save_json(dr, C.RESULTS / "dose_response.json")
            generalize.write_dose_response_macros(dr, PAPER / "dose_response_macros.tex")
        except ValueError as e:
            log.warning("dose-response analysis skipped: %s", e)

        log.info("cross-corpus generalisation: train on one corpus, evaluate cold on the other")
        cc_preds = models.cross_corpus_eval(X, y, groups, corpus, seeds=seeds,
                                            bands=bands, periods=periods)
        if cc_preds:
            cc = generalize.cross_corpus_report(cc_preds)
            evaluate.save_json(cc, C.RESULTS / "cross_corpus.json")
        else:
            log.warning("cross-corpus analysis skipped: only one corpus present in data/ "
                        "(smoke mode never has both -- run --real with both IoT-23 and "
                        "CTU-13 under data/ to exercise this)")

    # ---------------------------------------------------------- explainability
    # Exploratory, outside the pre-registered protocol -- see src/explain.py's
    # module docstring for why each analysis exists and what it can and
    # cannot claim to show.
    shap_band = shap_cross = None
    if args.explain:
        if not explain.HAVE_SHAP:
            log.warning("--explain given but shap is not installed "
                       "(pip install shap) -- explainability stage skipped")
        else:
            log.info("SHAP per-band attribution (%s)", "Gradient Boosting")
            shap_band = explain.explain_per_band(X, y, groups, bands, seed=seeds[0])
            if shap_band:
                evaluate.save_json(shap_band, C.RESULTS / "shap_band.json")
            if corpus is not None:
                log.info("SHAP cross-corpus attribution: native vs. transfer")
                shap_cross = explain.explain_cross_corpus(X, y, groups, corpus, seed=seeds[0])
                if shap_cross:
                    evaluate.save_json(shap_cross, C.RESULTS / "shap_cross_corpus.json")
            else:
                log.warning("--explain: cross-corpus attribution needs --extension "
                           "(for the corpus column) -- per-band SHAP still ran")
            explain.write_explain_macros(shap_band, shap_cross, PAPER / "explain_macros.tex")

    # ---------------------------------------------------------- diagnostics
    if args.skip_diagnostics:
        log.warning("--skip-diagnostics: importance, k-sweep and OOB curve omitted")
        fi, k_sweep, rf_oob = {"importance": None}, None, None
    else:
        log.info("permutation importance + model curves (the slow part; "
                 "use --skip-diagnostics to omit)")
        fi = models.fit_final_and_importance(X, y, groups, seed=seeds[0])
        k_sweep = models.knn_k_sweep(X, y, groups, seed=seeds[0])
        rf_oob = models.rf_oob_curve(X, y, groups, seed=seeds[0])

    figures.make_all(preds, results["per_band"],
                     channels["period"].to_numpy(), fi["importance"],
                     k_sweep, rf_oob, out=C.FIGURES,
                     watermark=args.smoke)
    if dr is not None:
        figures.fig_dose_response(dr, C.FIGURES)
    if cc is not None:
        figures.fig_cross_corpus(cc, C.FIGURES)
    if shap_band is not None:
        figures.fig_shap_band(shap_band, C.FIGURES)
    if shap_cross is not None:
        figures.fig_shap_cross_corpus(shap_cross, C.FIGURES)

    # ---------------------------------------------------------- summary
    print("\n" + "=" * 74)
    print(f"{'band':<6}{'detector':<16}{'DR':>8}{'FAR':>9}{'F1':>8}{'n':>9}")
    print("-" * 74)
    for b in C.BANDS:
        for m in C.DETECTORS:
            v = results["per_band"].get(m, {}).get(b)
            if v:
                print(f"{b:<6}{m:<16}{v['DR']:>8.3f}{v['FAR']:>9.4f}"
                      f"{v['F1']:>8.3f}{v['n']:>9d}")
    print("=" * 74)
    print(f"Breakdown criterion (Eq. 9 in the paper): DR < {C.TAU} or DR(B1) - DR > {C.DELTA}")
    for m, bd in results["breakdown"].items():
        bad = [b for b, v in bd.items() if v["breaks_down"]]
        print(f"  {m:<16} breaks down in: {', '.join(bad) if bad else 'no band'}")
    if dr is not None:
        print("\nDose-response crossing points (exploratory, not pre-registered):")
        for m, key in (("Random Forest", "rf"), ("Gradient Boosting", "gb"), ("k-NN", "knn"), ("Suricata", "sur")):
            if m not in dr["models"]:
                continue
            c = dr["models"][m]["crossing"]
            if c["mean"] is not None:
                print(f"  {m:<16} crosses tau={dr['tau']} near {c['mean']:.0f}s "
                      f"(range {c['range'][0]:.0f}-{c['range'][1]:.0f}s across "
                      f"{c['n_found']}/{c['n_seeds']} seeds)")
            else:
                print(f"  {m:<16} never crosses tau={dr['tau']} in the observed range "
                      f"-- itself a finding")
    if cc is not None:
        print("\nCross-corpus generalisation (exploratory, not pre-registered):")
        for direction, per_model in cc.items():
            print(f"  {direction.replace('_to_', ' -> ')}:")
            for m, bd in per_model.items():
                drs = ", ".join(f"{b}={bd[b]['DR']:.3f}" for b in C.BANDS if b in bd)
                print(f"    {m:<16} {drs}")
    if args.smoke:
        print("\n*** SMOKE MODE: synthetic data. These are NOT research results. ***")
    extra = ""
    if args.extension:
        extra = (", results/dose_response.json, results/cross_corpus.json, "
                 "paper/dose_response_macros.tex")
    if args.explain and (shap_band is not None or shap_cross is not None):
        extra += ", results/shap_band.json, results/shap_cross_corpus.json, paper/explain_macros.tex"
    print(f"\nWrote results/, figures/, paper/results_table.tex, "
          f"paper/results_macros.tex{extra}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tests/test_pipeline.py
"""Tests for the invariants the paper *claims*.

These are not decorative. Each one guards a specific sentence in the
manuscript; if a test fails, a claim in the paper has become false.

    pytest -q
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src import config as C
from src import evaluate, features, generalize, intervals, models, preprocess
from src.synth import write_synthetic_conn_log


# --------------------------------------------------------------------------
# Section III-E: interval derivation, Eq. (2)-(4)
# --------------------------------------------------------------------------
def test_inter_arrival_is_sorted_diff():
    t = np.array([300.0, 0.0, 60.0, 120.0])       # deliberately unsorted
    assert np.allclose(intervals.inter_arrival(t), [60, 60, 180])


def test_mad_matches_definition():
    x = np.array([10.0, 12.0, 10.0, 14.0, 10.0])
    assert intervals.median_absolute_deviation(x) == pytest.approx(0.0, abs=1e-12)
    y = np.array([1.0, 2.0, 3.0, 4.0, 100.0])
    assert intervals.median_absolute_deviation(y) == pytest.approx(1.0)


def test_median_is_robust_to_a_long_gap():
    """The reason the paper uses median, not mean: one missed check-in must not
    move the estimated period."""
    clean = np.arange(0, 20 * 60 + 1, 60, dtype=float)
    gapped = np.delete(clean, 10)                  # one skipped beacon
    assert intervals.channel_statistics(clean)["period"] == pytest.approx(60.0)
    assert intervals.channel_statistics(gapped)["period"] == pytest.approx(60.0)
    # the mean, by contrast, is dragged upward
    assert intervals.channel_statistics(gapped)["dt_mean"] > 60.0


def test_jitter_ratio_is_mad_over_period():
    st = intervals.channel_statistics(np.array([0, 100, 200, 300, 400], dtype=float))
    assert st["jitter_ratio"] == pytest.approx(st["dt_mad"] / st["period"])


def test_band_boundaries_are_exact():
    lo, hi = C.BAND_EDGES_S
    assert intervals.assign_band(lo - 1e-9) == "B1"
    assert intervals.assign_band(lo) == "B2"        # boundary is inclusive-low
    assert intervals.assign_band(hi - 1e-9) == "B2"
    assert intervals.assign_band(hi) == "B3"
    assert intervals.assign_band(float("nan")) == "unknown"


# --------------------------------------------------------------------------
# Section III-D: class construction
# --------------------------------------------------------------------------
@pytest.fixture(scope="module")
def synthetic(tmp_path_factory):
    p = tmp_path_factory.mktemp("smoke") / "conn.log.labeled"
    write_synthetic_conn_log(p, n_c2_channels=30, n_benign_channels=60)
    return p


def test_background_traffic_is_excluded_not_assumed_benign(synthetic):
    raw = preprocess.read_conn_log(synthetic)
    labelled = preprocess.assign_class(raw)
    bg = labelled[labelled.iloc[:, -2].astype(str).str.contains("Background", na=False)] \
        if "Background" in raw.to_string()[:0] else None      # structural guard
    # every Background-Unknown row must have y == NaN
    lab_col = [c for c in labelled.columns if labelled[c].astype(str)
               .str.contains("Background-Unknown", na=False).any()]
    assert lab_col, "synthetic data should contain background rows"
    mask = labelled[lab_col[0]].astype(str).str.contains("Background-Unknown", na=False)
    assert labelled.loc[mask, "y"].isna().all()


def test_min_observation_filter_reports_what_it_dropped(synthetic):
    raw = preprocess.read_conn_log(synthetic)
    df = preprocess.assign_class(raw).dropna(subset=["y"])
    df["y"] = df["y"].astype(int)
    df = preprocess.build_channels(df)
    kept, rep = preprocess.filter_min_observations(df, min_obs=10)
    assert rep["channels_kept"] + rep["channels_dropped"] == rep["channels_before"]
    assert kept["channel_id"].value_counts().min() >= 10


# --------------------------------------------------------------------------
# Section III-F: the band label must never reach the model
# --------------------------------------------------------------------------
def test_band_label_never_enters_the_feature_matrix(synthetic):
    flows, _ = preprocess.prepare([synthetic])
    flows, _ = intervals.derive(flows)
    X, y, groups, bands = features.assemble(flows)
    for forbidden in C.FORBIDDEN_FEATURES:
        assert forbidden not in X.columns


def test_forbidden_feature_raises(synthetic):
    df = pd.DataFrame({"duration": [1.0], "band": ["B1"], "y": [0]})
    C.ZEEK_NUMERIC.append("band")
    try:
        with pytest.raises(AssertionError):
            features.feature_columns(df)
    finally:
        C.ZEEK_NUMERIC.remove("band")


# --------------------------------------------------------------------------
# Section III-H: no channel may appear in both train and test
# --------------------------------------------------------------------------
def test_groupkfold_never_leaks_a_channel(synthetic):
    from sklearn.model_selection import GroupKFold
    flows, _ = preprocess.prepare([synthetic])
    flows, _ = intervals.derive(flows)
    X, y, groups, _ = features.assemble(flows)
    gkf = GroupKFold(n_splits=C.N_FOLDS)
    for tr, te in gkf.split(X, y, groups=groups):
        assert not (set(groups[tr]) & set(groups[te]))
        models._assert_no_group_overlap(groups, tr, te)   # the runtime guard


def test_leakage_guard_actually_fires():
    groups = np.array(["a", "a", "b", "b"])
    with pytest.raises(AssertionError):
        models._assert_no_group_overlap(groups, np.array([0, 1, 2]), np.array([1, 3]))


# --------------------------------------------------------------------------
# Section III-I: metrics and the pre-registered breakdown criterion
# --------------------------------------------------------------------------
def test_metrics_against_a_hand_computed_case():
    y = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
    s = np.array([.9, .8, .7, .2, .9, .1, .1, .1, .1, .1])
    m = evaluate.metrics(y, s, threshold=0.5)
    assert (m["tp"], m["fp"], m["tn"], m["fn"]) == (3, 1, 5, 1)
    assert m["DR"] == pytest.approx(3 / 4)
    assert m["FAR"] == pytest.approx(1 / 6)
    assert m["P"] == pytest.approx(3 / 4)
    assert m["F1"] == pytest.approx(0.75)


def test_far_is_nan_when_a_band_has_no_benign_flows():
    """A band with no negatives cannot support a false-alarm estimate. The code
    must return NaN rather than silently reporting zero."""
    m = evaluate.metrics(np.array([1, 1, 1]), np.array([.9, .9, .1]))
    assert np.isnan(m["FAR"])


def test_breakdown_criterion_applies_both_clauses():
    bm = {"B1": {"DR": 0.95}, "B2": {"DR": 0.90}, "B3": {"DR": 0.60}}
    bd = evaluate.breakdown(bm, tau=0.70, delta=0.20)
    assert bd["B1"]["breaks_down"] is False
    assert bd["B2"]["breaks_down"] is False          # 0.05 drop, above tau
    assert bd["B3"]["below_tau"] and bd["B3"]["drop_exceeds_delta"]
    # a detector above tau can still break down on the relative clause
    bd2 = evaluate.breakdown({"B1": {"DR": 0.99}, "B2": {"DR": 0.75}},
                             tau=0.70, delta=0.20)
    assert bd2["B2"]["breaks_down"] and not bd2["B2"]["below_tau"]


def test_thresholds_are_pre_registered_constants():
    """Guards against someone tuning tau after seeing results."""
    assert C.TAU == 0.70 and C.DELTA == 0.20


# --------------------------------------------------------------------------
# CTU-13 binetflow path -- Argus flows, no Zeek and no pcaps required
# --------------------------------------------------------------------------
def _tiny_binetflow(tmp, period_s, n=30):
    """One perfectly periodic CC channel at a known period."""
    import pandas as pd
    t0 = pd.Timestamp("2011-08-10 09:46:59.607825")
    rows = []
    for i in range(n):
        ts = t0 + pd.Timedelta(seconds=period_s * i)
        rows.append([ts.strftime("%Y/%m/%d %H:%M:%S.%f"), 0.5, "tcp",
                     "147.32.84.165", 1234, "  ->", "212.117.171.138", 443,
                     "CON", 0, 0, 8, 900, 400,
                     "flow=From-Botnet-V42-TCP-Established-CC1"])
    cols = ["StartTime","Dur","Proto","SrcAddr","Sport","Dir","DstAddr","Dport",
            "State","sTos","dTos","TotPkts","TotBytes","SrcBytes","Label"]
    p = tmp / "capture.binetflow"
    pd.DataFrame(rows, columns=cols).to_csv(p, index=False)
    return p


def test_binetflow_timestamps_are_epoch_seconds(tmp_path_factory):
    """Regression guard.

    pandas 2.x gives datetime64[ns] and pandas 3.x gives datetime64[us], so
    converting via .astype('int64') yields periods that are 1000x wrong on one
    of the two -- silently. This asserts the derived period equals the period
    the data was generated with, which catches any unit error.
    """
    import numpy as np
    from src import preprocess, intervals
    p = _tiny_binetflow(tmp_path_factory.mktemp("ctu"), period_s=300.0)
    flows, _ = preprocess.prepare([p])
    _, channels = intervals.derive(flows)
    assert len(channels) == 1
    assert channels["period"].iloc[0] == pytest.approx(300.0, abs=0.01)
    assert channels["band"].iloc[0] == "B2"


def test_binetflow_bands_match_generated_periods(tmp_path_factory):
    from src import preprocess, intervals
    for period, expected in ((20.0, "B1"), (300.0, "B2"), (1800.0, "B3")):
        p = _tiny_binetflow(tmp_path_factory.mktemp("ctu"), period_s=period)
        flows, _ = preprocess.prepare([p])
        _, ch = intervals.derive(flows)
        assert ch["band"].iloc[0] == expected, f"period {period}s -> {ch['band'].iloc[0]}"


def test_ctu13_non_cc_botnet_traffic_is_excluded(tmp_path_factory):
    """CTU-13's From-Botnet family covers spam and DDoS as well as C2. Only
    CC-bearing flows are positives; the rest must be dropped, never called
    benign -- that would corrupt the false-alarm denominator."""
    import pandas as pd
    from src import preprocess
    tmp = tmp_path_factory.mktemp("ctu_excl")
    cols = ["StartTime","Dur","Proto","SrcAddr","Sport","Dir","DstAddr","Dport",
            "State","sTos","dTos","TotPkts","TotBytes","SrcBytes","Label"]
    base = ["2011/08/10 09:47:33.086301", 0.5, "tcp", "1.1.1.1", 1234, "  ->",
            "2.2.2.2", 443, "CON", 0, 0, 8, 900, 400]
    rows = [base + ["flow=From-Botnet-V42-TCP-Established-CC1"],
            base + ["flow=From-Botnet-V42-TCP-Attempt-SPAM"],
            base + ["flow=Normal-V42-Jist"],
            base + ["flow=Background-UDP-Established"]]
    p = tmp / "x.binetflow"
    pd.DataFrame(rows, columns=cols).to_csv(p, index=False)
    df = preprocess.assign_class(preprocess.read_any(p))
    y = df["y"].tolist()
    assert y[0] == 1.0                     # CC -> positive
    assert pd.isna(y[1])                   # botnet non-CC -> excluded
    assert y[2] == 0.0                     # normal -> benign
    assert len(df) == 3                    # background dropped during the read


def test_mixed_corpora_do_not_silently_drop_one_of_them(tmp_path_factory):
    """IoT-23 names its label column differently from CTU-13. If classes were
    assigned after concatenation, one corpus would land in a column that is NaN
    for the other's rows and be dropped as unlabelled -- with no error. The
    cross-corpus claim in the paper depends on this not happening."""
    tmp = tmp_path_factory.mktemp("mixed")
    zeek = tmp / "conn.log.labeled"
    write_synthetic_conn_log(zeek, n_c2_channels=12, n_benign_channels=12)
    argus = _tiny_binetflow(tmp, period_s=300.0, n=40)

    only_zeek, _ = preprocess.prepare([zeek])
    only_argus, _ = preprocess.prepare([argus])
    both, _ = preprocess.prepare([zeek, argus])

    assert len(only_zeek) > 0 and len(only_argus) > 0
    assert len(both) == len(only_zeek) + len(only_argus), \
        "mixing the corpora lost rows -- a label column was not read"
    assert both["source_capture"].nunique() == 2


def test_smoke_figures_are_watermarked(tmp_path_factory):
    """A figure drawn from synthetic data must say so on its face. Shipping an
    unwatermarked smoke-run plot is how a fabricated figure reaches a
    submission."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from src import figures

    out = tmp_path_factory.mktemp("figs")

    def draw(watermark):
        figures.WATERMARK = watermark
        try:
            fig, ax = plt.subplots()
            ax.plot([0, 1], [0, 1])
            stamped = [t.get_text() for t in fig.texts]
            figures._save(fig, f"probe_{int(watermark)}", out)
        finally:
            figures.WATERMARK = False
        return (out / f"probe_{int(watermark)}.png").read_bytes()

    off = draw(False)
    on = draw(True)
    assert on != off, "watermark did not change the rendered figure"

    # and the stamp itself carries the words a reader must see
    fig, _ = plt.subplots()
    art = figures._stamp(fig)
    assert art.get_text() == "SYNTHETIC DATA"
    plt.close(fig)


# --------------------------------------------------------------------------
# src/generalize.py -- dose-response and cross-corpus extension
# --------------------------------------------------------------------------
def test_corpus_tagging_from_file_dispatch(tmp_path_factory):
    """assemble_ext's train/test split for cross-corpus eval depends entirely
    on preprocess.prepare tagging each row correctly. A .binetflow file must
    be tagged 'ctu13' and a conn.log.labeled 'iot23', by file type -- not by
    a filename guess that a differently-named real download would break."""
    tmp = tmp_path_factory.mktemp("tag")
    zeek = tmp / "conn.log.labeled"
    write_synthetic_conn_log(zeek, n_c2_channels=8, n_benign_channels=8)
    argus = _tiny_binetflow(tmp, period_s=120.0)

    flows, _ = preprocess.prepare([zeek, argus])
    assert set(flows["corpus"].unique()) == {"iot23", "ctu13"}
    assert (flows.loc[flows["source_capture"] == "capture", "corpus"] == "ctu13").all()
    assert (flows.loc[flows["source_capture"] == zeek.stem, "corpus"] == "iot23").all()


def test_fine_bin_edges_are_quantile_spaced_and_monotonic():
    rng = np.random.default_rng(0)
    periods = np.exp(rng.uniform(np.log(5), np.log(3000), 200))
    edges = generalize.fine_bin_edges(periods, n_bins=6)
    assert np.all(np.diff(edges) > 0), "bin edges must be strictly increasing"
    assert edges[0] <= periods.min() and edges[-1] >= periods.max()


def test_fine_bin_edges_rejects_too_few_channels():
    with pytest.raises(ValueError):
        generalize.fine_bin_edges(np.array([10.0, 20.0, 30.0]), n_bins=8)


def test_assign_fine_bin_excludes_out_of_range_rather_than_clipping():
    edges = np.array([10.0, 100.0, 1000.0])
    period = np.array([5.0, 10.0, 50.0, 999.0, 5000.0, np.nan])
    idx = generalize.assign_fine_bin(period, edges)
    # 5.0 is below edges[0] and 5000.0 is above edges[-1]: both excluded (-1),
    # not silently folded into the nearest end bin.
    assert idx.tolist() == [-1, 0, 0, 1, -1, -1]


def test_find_crossings_matches_hand_computed_linear_interpolation():
    # A straight line from (0, 1.0) to (10, 0.0) crosses 0.5 exactly at x=5.
    x = np.array([0.0, 10.0])
    y = np.array([1.0, 0.0])
    assert generalize.find_crossings(x, y, 0.5) == [5.0]


def test_find_crossings_handles_non_monotonic_curves():
    """k-NN's dose-response curve in this study is not monotonic -- it dips
    and partially recovers. The crossing finder must report every sign
    change, not just the first, or a recovery back above tau would be
    silently discarded."""
    x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
    y = np.array([0.9, 0.3, 0.3, 0.9, 0.9])   # crosses tau=0.5 going down, then up
    crossings = generalize.find_crossings(x, y, 0.5)
    assert len(crossings) == 2
    assert crossings[0] < 1.0 and 2.0 < crossings[1] < 3.0


def test_crossing_point_per_seed_never_found_reports_zero_not_extrapolated():
    """Random Forest never crosses tau in this study's real run. That must
    surface as n_found=0 with mean=None -- never as an extrapolated number
    manufactured from a curve that never actually reaches tau."""
    bin_metrics = {
        0: {"DR_per_seed": [0.95, 0.94]},
        1: {"DR_per_seed": [0.92, 0.93]},
    }
    centers = np.array([10.0, 100.0])
    result = generalize.crossing_point_per_seed(bin_metrics, centers, tau=0.70)
    assert result["n_found"] == 0
    assert result["mean"] is None
    assert result["range"] is None
    assert result["n_seeds"] == 2


def _multi_channel_binetflow(tmp, n_cc=6, n_benign=6, n_obs=15):
    """Several CTU-13 channels of both classes -- enough groups per class for
    the internal GroupKFold hyperparameter search cross_corpus_eval runs on
    whichever corpus is training in a given direction."""
    t0 = pd.Timestamp("2011-08-10 09:46:59.607825")
    rows = []
    cols = ["StartTime", "Dur", "Proto", "SrcAddr", "Sport", "Dir", "DstAddr",
            "Dport", "State", "sTos", "dTos", "TotPkts", "TotBytes", "SrcBytes",
            "Label"]
    for ci in range(n_cc):
        period = 60.0 + ci * 20
        for i in range(n_obs):
            ts = t0 + pd.Timedelta(seconds=period * i)
            rows.append([ts.strftime("%Y/%m/%d %H:%M:%S.%f"), 0.5, "tcp",
                        f"147.32.84.{165 + ci}", 1234, "  ->",
                        "212.117.171.138", 443, "CON", 0, 0, 8, 900, 400,
                        "flow=From-Botnet-V42-TCP-Established-CC1"])
    for bi in range(n_benign):
        period = 60.0 + bi * 20
        for i in range(n_obs):
            ts = t0 + pd.Timedelta(seconds=period * i)
            rows.append([ts.strftime("%Y/%m/%d %H:%M:%S.%f"), 0.5, "tcp",
                        f"147.32.85.{10 + bi}", 5555, "  ->",
                        "8.8.8.8", 53, "CON", 0, 0, 8, 300, 150,
                        "flow=To-Normal-UDP-CVUT"])
    p = tmp / "capture.binetflow"
    pd.DataFrame(rows, columns=cols).to_csv(p, index=False)
    return p


def test_cross_corpus_eval_never_lets_a_channel_appear_on_both_sides(tmp_path_factory):
    """The whole point of the cross-corpus check is a cold evaluation: a
    channel that trained the model must never also be scored. Since a
    channel's five-tuple belongs to exactly one capture file, this should
    hold by construction -- but models.cross_corpus_eval asserts it anyway,
    and this test exercises that assertion actually runs and passes."""
    tmp = tmp_path_factory.mktemp("cc")
    zeek = tmp / "conn.log.labeled"
    write_synthetic_conn_log(zeek, n_c2_channels=20, n_benign_channels=20)
    argus = _multi_channel_binetflow(tmp)

    flows, _ = preprocess.prepare([zeek, argus])
    flows, channels = intervals.derive(flows)
    X, y, groups, bands, periods, corpus = features.assemble_ext(flows)
    if len(np.unique(y)) < 2 or "ctu13" not in corpus or "iot23" not in corpus:
        pytest.skip("fixture did not produce both classes in both corpora")

    out = models.cross_corpus_eval(X, y, groups, corpus, seeds=(11,),
                                   bands=bands, periods=periods)
    for direction, per_model in out.items():
        for model_name, pred in per_model.items():
            # Every predicted flow's group must be disjoint from every
            # training-side group -- re-derive both sides and check directly.
            train_corpus, test_corpus = direction.split("_to_")
            train_groups = set(groups[corpus == train_corpus])
            test_groups = set(groups[corpus == test_corpus])
            assert train_groups.isdisjoint(test_groups), \
                f"{direction}/{model_name}: train and test corpora share a channel"
    assert isinstance(out, dict)
    assert len(out) == 2, "expected both cross-corpus directions to run"

---
## Step 2b — Prove it works before downloading anything

The smoke test runs the entire pipeline on synthetic data in about a minute. If this
fails, stop and fix it — do not start downloading 20 scenarios first.

**Nothing the smoke test produces is a research result.** It exists only to verify
plumbing.

In [ ]:
!python run_all.py --smoke --seeds 2 2>&1 | tail -25

Now the test suite. Each test guards a specific claim the paper makes — that the median
survives a missed check-in, that no channel leaks between train and test, that the band
label never reaches the model, that τ and δ have not been retuned after the fact.

**If any test fails, a sentence in your paper has become false.**

In [ ]:
!pip install -q pytest
!python -m pytest -q tests/

---
## Step 3 — Download IoT-23 scenarios

IoT-23 is published per scenario at
`https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/IndividualScenarios/`.

The list below mixes **malware captures** (your positive class) with the three
**honeypot captures** (your benign class). You need both: without benign traffic the
false-alarm rate is undefined and half the paper's evaluation collapses.

The cell checks each file's size before downloading and skips anything above
`MAX_MB`. Some IoT-23 captures are enormous (hundreds of MB to over a GB of log);
the cap protects your session. Raise it if you have Colab Pro and patience.

In [ ]:
BASE = 'https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/IndividualScenarios'

# Malware captures (positive class) + honeypot captures (benign class).
# Start with this set; widen it once you see the label census in Step 4.
SCENARIOS = [
    'CTU-IoT-Malware-Capture-8-1',
    'CTU-IoT-Malware-Capture-20-1',
    'CTU-IoT-Malware-Capture-21-1',
    'CTU-IoT-Malware-Capture-33-1',
    'CTU-IoT-Malware-Capture-42-1',
    'CTU-IoT-Malware-Capture-44-1',
    'CTU-IoT-Malware-Capture-49-1',
    'CTU-IoT-Malware-Capture-60-1',
    'CTU-Honeypot-Capture-4-1',      # benign
    'CTU-Honeypot-Capture-5-1',      # benign
    'CTU-Honeypot-Capture-7-1',      # benign
]

MAX_MB = 400          # skip logs larger than this
GET_PCAPS = True      # needed only for the Suricata baseline (Step 5)
MAX_PCAP_MB = 300

In [ ]:
import re, urllib.request, urllib.error, pathlib, html

def head_size(url):
    try:
        r = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(r, timeout=30) as resp:
            n = resp.headers.get('Content-Length')
            return int(n) if n else None
    except Exception:
        return None

def listdir(url):
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            page = resp.read().decode('utf-8', 'replace')
        return [html.unescape(m) for m in re.findall(r'href="([^"?][^"]*)"', page)]
    except Exception as e:
        print('  ! could not list', url, e); return []

def fetch(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f'  = already have {dest.name} ({dest.stat().st_size/1e6:.1f} MB)'); return True
    try:
        urllib.request.urlretrieve(url, dest)
        print(f'  + {dest.name} ({dest.stat().st_size/1e6:.1f} MB)'); return True
    except Exception as e:
        print(f'  ! failed {url}: {e}'); return False

got_logs = got_pcaps = 0
for s in SCENARIOS:
    print(f'\n{s}')
    log_url = f'{BASE}/{s}/bro/conn.log.labeled'
    size = head_size(log_url)
    if size is None:
        print('  ? size unknown, attempting anyway')
    elif size/1e6 > MAX_MB:
        print(f'  - SKIP log: {size/1e6:.0f} MB exceeds MAX_MB={MAX_MB}'); continue
    if fetch(log_url, pathlib.Path(f'data/iot23/{s}/conn.log.labeled')):
        got_logs += 1

    if GET_PCAPS:
        pcaps = [h for h in listdir(f'{BASE}/{s}/') if h.lower().endswith('.pcap')]
        for pc in pcaps[:1]:
            purl = f'{BASE}/{s}/{pc}'
            psize = head_size(purl)
            if psize and psize/1e6 > MAX_PCAP_MB:
                print(f'  - SKIP pcap: {psize/1e6:.0f} MB > MAX_PCAP_MB'); continue
            if fetch(purl, pathlib.Path(f'data/iot23/{s}/{pc}')):
                got_pcaps += 1

print(f'\n=== downloaded {got_logs} connection logs, {got_pcaps} pcaps ===')
if got_logs == 0:
    print('NOTHING DOWNLOADED — check your network, or the host may be temporarily down.')

---
## Step 3b — Download CTU-13 scenarios (cross-corpus validation)

CTU-13's thirteen scenarios live at
`https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-<N>/`, numbered
42 through 54. The file you want in each is under `detailed-bidirectional-flow-labels/`
and ends in `.binetflow` — labelled Argus flows, between roughly 70 MB and 370 MB per
scenario. **No pcaps. No Zeek.**

Three scenarios are enough for a cross-corpus check and keep the download honest. Widen
`CTU_SCENARIOS` if the Step 4 census shows a band thin on CTU-13 channels.

A note on labels: CTU-13's `From-Botnet` family covers spam, DDoS and scanning as well
as command and control. Only the `CC`-bearing labels count as positives here. Botnet
traffic that is not C2 is **excluded**, never relabelled benign — calling a DDoS flow
benign would corrupt the false-alarm denominator. There is a unit test for this.

In [ ]:
CTU_BASE = 'https://mcfp.felk.cvut.cz/publicDatasets'

# CTU-13 scenario numbers. The full set is 42..54.
CTU_SCENARIOS = [42, 43, 47]
MAX_CTU_MB = 400

got_ctu = 0
for n in CTU_SCENARIOS:
    d = f'CTU-Malware-Capture-Botnet-{n}'
    listing = f'{CTU_BASE}/{d}/detailed-bidirectional-flow-labels/'
    print(f'\n{d}')
    names = [h for h in listdir(listing) if h.endswith('.binetflow')
             or h.endswith('.binetflow.labeled')]
    if not names:
        print('  ! no .binetflow found — check the scenario number'); continue
    # prefer the canonical .binetflow; it is the file CTU-13 is distributed as
    names.sort(key=lambda x: (not x.endswith('.binetflow'), x))
    fn = names[0]
    url = listing + fn
    size = head_size(url)
    if size and size/1e6 > MAX_CTU_MB:
        print(f'  - SKIP: {size/1e6:.0f} MB > MAX_CTU_MB={MAX_CTU_MB}'); continue
    if size:
        print(f'  downloading {fn} ({size/1e6:.0f} MB)')
    if fetch(url, pathlib.Path(f'data/ctu13/{d}/{fn}')):
        got_ctu += 1

print(f'\n=== {got_ctu} CTU-13 scenario(s) downloaded ===')
if got_ctu == 0:
    print('No CTU-13 data. The pipeline still runs on IoT-23 alone, but then you MUST')
    print('remove the cross-corpus claim from the paper — see Step 9.')

Sanity-check one file before trusting it: the read must produce epoch-second
timestamps, or every derived period is wrong by a factor of a thousand and every
channel collapses into $B_1$. This exact bug is why `read_binetflow` avoids
`.astype('int64')`.

In [ ]:
import sys, pathlib
sys.path.insert(0, '/content/quiet-signal')
from src import preprocess

bf = sorted(pathlib.Path('data/ctu13').rglob('*.binetflow*'))
if not bf:
    print('no CTU-13 files to check')
else:
    d = preprocess.read_binetflow(bf[0])
    print(f'{bf[0].name}: {len(d):,} non-background flows')
    print('ts range (epoch s):', d['ts'].min(), '->', d['ts'].max())
    import datetime as _dt
    print('  i.e.', _dt.datetime.fromtimestamp(d['ts'].min(), _dt.timezone.utc), 'to',
          _dt.datetime.fromtimestamp(d['ts'].max(), _dt.timezone.utc))
    print('  (CTU-13 was captured in 2011 — if you see 1970, the units are wrong)')
    print('\nlabel sample:'); print(d['label'].value_counts().head(8))

---
## Step 4 — Check what you actually got

Before modelling, look at the labels. This is the single most useful diagnostic in the
whole notebook, and it answers three questions your examiner will ask:

1. **Do you have C2 / heartbeat flows at all?** No positives means nothing to detect.
2. **Do you have benign flows?** No negatives means false-alarm rate is undefined.
3. **Are all three interval bands populated, in both classes?** A band with only one
   class cannot support a per-band rate.

If any answer is no, widen `SCENARIOS` in Step 3 and re-run.

In [ ]:
import sys, collections, pandas as pd
sys.path.insert(0, '/content/quiet-signal')
from src import preprocess, intervals
from pathlib import Path

logs  = sorted(Path('data').rglob('conn.log.labeled'))
flows_ctu = sorted(Path('data').rglob('*.binetflow*'))
logs = logs + flows_ctu
print(f'{len(logs)} source file(s): {len(logs)-len(flows_ctu)} IoT-23, {len(flows_ctu)} CTU-13\n')

for p in logs:
    d = preprocess.read_any(p)
    col = preprocess._label_column(d)
    census = collections.Counter(d[col].astype(str).str.strip())
    print(f'{p.parent.name}/{p.name}  ({len(d):,} flows)')
    for lab, n in census.most_common(6):
        print(f'    {n:>9,}  {lab[:66]}')
    print()

In [ ]:
# Full stage B + C, then the band/class cross-tab that decides feasibility
flows, filt = preprocess.prepare(logs)
flows, channels = intervals.derive(flows)

print('FILTER REPORT'); [print(f'  {k:22} {v:,}') for k, v in filt.items()]
print()
ct = pd.crosstab(channels['band'], channels['y_channel'])
ct.columns = ['benign' if c == 0 else 'C2' for c in ct.columns]
print('CHANNELS PER BAND AND CLASS'); print(ct)
print()
bad = [b for b in ('B1','B2','B3')
       if b not in ct.index or (ct.loc[b] == 0).any()]
if bad:
    print(f'!! Bands missing a class: {bad}')
    print('   Per-band FAR or DR will be undefined there. Add scenarios in Step 3/3b.')
else:
    print('OK — every band has both classes. You can compute all per-band metrics.')

---
## Step 5 — Suricata signature baseline *(optional but load-bearing)*

**Skip this and you lose a contribution.** The paper's second claim is a
signature-versus-ML comparison. Without Suricata you still have the interval-stratified
analysis, but you must remove the comparison claim from the abstract, Section IV-A and
the conclusion.

This installs Suricata, fetches the Emerging Threats Open ruleset, and replays each
pcap. The ruleset is deliberately **not tuned** to the traffic — tuning a rule to the
captures you then evaluate on would make the comparison circular.

In [ ]:
!apt-get -qq update && apt-get -qq install -y suricata > /dev/null 2>&1
!suricata -V 2>/dev/null || echo 'Suricata not installed'

In [ ]:
import pathlib, urllib.request
RULES_URL = 'https://rules.emergingthreats.net/open/suricata-7.0/emerging-all.rules'
rules = pathlib.Path('data/rules'); rules.mkdir(parents=True, exist_ok=True)
rf = rules / 'emerging-all.rules'

if rf.exists() and rf.stat().st_size > 0:
    print(f'ruleset already present: {rf.stat().st_size/1e6:.1f} MB')
else:
    try:
        urllib.request.urlretrieve(RULES_URL, rf)
        print(f'ruleset downloaded: {rf.stat().st_size/1e6:.1f} MB')
    except Exception as e:
        print('! could not fetch the ET Open ruleset:', e)
        print('  Without it the Suricata baseline is skipped and you must remove the')
        print('  signature-vs-ML comparison claim from the paper.')

In [ ]:
from pathlib import Path
import subprocess
pcaps = sorted(Path('data').rglob('*.pcap'))
print(f'{len(pcaps)} pcap(s) to replay\n')
for pc in pcaps:
    out = pc.parent
    if (out/'eve.json').exists():
        print('= already done', pc.parent.name); continue
    print('replaying', pc.name, f'({pc.stat().st_size/1e6:.1f} MB)')
    subprocess.run(['suricata','-r',str(pc),'-S','data/rules/emerging-all.rules',
                    '-l',str(out)], capture_output=True)
    ev = out/'eve.json'
    print('  ->', 'eve.json', f'{ev.stat().st_size/1e6:.1f} MB' if ev.exists() else 'FAILED')

eves = sorted(Path('data').rglob('eve.json'))
print(f'\n{len(eves)} eve.json produced — the pipeline will pick these up automatically.')

---
## Step 6 — Run the real pipeline

This is the run that produces your paper's numbers. Five seeds × five folds, with
nested tuning inside each training fold. Expect several minutes; longer if you added
many scenarios.

**Timing.** On the default scenario set expect roughly 3–6 minutes. Knobs if your
runtime is tight:

- `--seeds 2` — a quick sanity pass. Use all five seeds for the numbers you report.
- `--skip-diagnostics` — omits permutation importance, the k-sweep and the OOB curve.
  Your three main results figures are still produced; you lose Figs. of importance,
  k-sweep and OOB. That is the slowest third of the run.
- `--extension` — also runs the exploratory dose-response and cross-corpus analyses
  (see Step 6b below for what that gets you). It adds one more training pass on top
  of the main run, so it is off by default here; flip `RUN_EXTENSION` to `True` to
  include it in this same run rather than re-running the whole pipeline a second time.

In [ ]:
RUN_EXTENSION = True   # dose-response curve + cross-corpus generalisation (Step 6b)
RUN_EXPLAIN = True     # SHAP per-band + cross-corpus attribution (Step 6c, needs `pip install shap`)
flag = ' --extension' if RUN_EXTENSION else ''
flag += ' --explain' if RUN_EXPLAIN else ''
# Full run. Add --skip-diagnostics if the runtime is time-limited.
!python run_all.py --real{flag} 2>&1 | tail -80

---
## Step 6b — The dose-response and cross-corpus results (exploratory)

These come from the same run above if `RUN_EXTENSION` was `True`. Both analyses sit
**outside** the pre-registered breakdown criterion (τ, δ, fixed in `src/config.py`
before any detector ran) — they were not fixed in advance because they did not exist
until now. Report them as exploratory findings, not as confirmatory tests, and say so
explicitly in the paper.

- **Dose-response curve.** Where exactly does each detector cross τ = 0.70, on a
  continuous period axis rather than the three fixed bands? A detector that never
  crosses (as Random Forest does not, in the original run) reports `n_found=0` —
  that is the finding, not a gap to fill in with an extrapolated number.
- **Cross-corpus generalisation.** Train on one corpus in full, evaluate cold on the
  other, both directions. This is the honest test of whether a detector learned
  something about beaconing, or just about IoT-23 (or CTU-13) specifically. It only
  runs if both corpora are present under `data/` — Step 3b downloaded CTU-13.

In [ ]:
import json, pathlib
dr_path = pathlib.Path('results/dose_response.json')
cc_path = pathlib.Path('results/cross_corpus.json')

if dr_path.exists():
    dr = json.load(open(dr_path))
    print(f"Dose-response curve: {len(dr['centers'])} bins, tau={dr['tau']}\n")
    for m, d in dr['models'].items():
        c = d['crossing']
        if c['mean'] is not None:
            print(f"  {m:16} crosses tau near {c['mean']:.0f}s "
                  f"(range {c['range'][0]:.0f}-{c['range'][1]:.0f}s, "
                  f"{c['n_found']}/{c['n_seeds']} seeds found a crossing)")
        else:
            print(f'  {m:16} never crosses tau in the observed range -- itself a finding')
else:
    print('No dose_response.json -- set RUN_EXTENSION = True in Step 6 and re-run.')

print()
if cc_path.exists():
    cc = json.load(open(cc_path))
    if cc:
        for direction, per_model in cc.items():
            print(f"{direction.replace('_to_', ' -> ')}:")
            for m, bd in per_model.items():
                drs = ', '.join(f"{b}={bd[b]['DR']:.3f}" for b in ('B1','B2','B3') if b in bd)
                print(f'  {m:16} {drs}')
    else:
        print('cross_corpus.json is empty -- only one corpus was present under data/.')
        print('Run Step 3b to download CTU-13, or the cross-corpus claim cannot be made.')
else:
    print('No cross_corpus.json -- set RUN_EXTENSION = True in Step 6 and re-run.')

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for f in ['fig_dose_response', 'fig_cross_corpus']:
    p = Path('figures')/f'{f}.png'
    if p.exists():
        print(f'\n=== {f} ==='); display(Image(str(p)))

---
## Step 6c — SHAP explainability (exploratory)

Also outside the pre-registered protocol, same caveat as Step 6b. Two views of
the same fitted Gradient Boosting model:

- **Per-band attribution.** Mean |SHAP| for the top features, computed
  separately within each band. A feature that grows or shrinks across the
  three bars is a candidate mechanism for the dose-response curve above --
  not just another restatement of the DR numbers.
- **Cross-corpus attribution.** The same fitted model scored on a held-out
  slice of its own training corpus ("native") vs. the entire other corpus
  ("transfer"). A feature whose bar height changes between the two is a
  candidate explanation for *why* cross-corpus detection collapses, which
  Step 6b's numbers show happens but cannot say why.


In [ ]:
from IPython.display import Image, display
from pathlib import Path
for f in ['fig_shap_band', 'fig_shap_cross_corpus']:
    p = Path('figures')/f'{f}.png'
    if p.exists():
        print(f'\n=== {f} ==='); display(Image(str(p)))
    else:
        print(f'{f}: not produced (needs --explain, and shap installed; '
              f'fig_shap_cross_corpus additionally needs --extension for both corpora)')


---
## Step 7 — Look at the results before you believe them

Three checks, in order of how badly they would embarrass you:

1. **Does detection actually degrade with interval?** The paper asserts it does. If your
   data says otherwise, **rewrite Section V to report what you observed.** A null result
   honestly reported is publishable; a narrative that survives contradicting data is not.
2. **Is any FAR implausibly high?** If so, check the decision threshold against your
   score distribution before concluding anything about the detector.
3. **Are band populations large enough** to support the rates you are quoting?

In [ ]:
import json, pandas as pd
res = json.load(open('results/results.json'))
rows = []
for model, bands in res['per_band'].items():
    for b, m in bands.items():
        rows.append(dict(band=b, detector=model, DR=round(m['DR'],3),
                         DR_sd=round(m.get('DR_sd',0),3), FAR=round(m['FAR'],4),
                         F1=round(m['F1'],3), n=m['n']))
df = pd.DataFrame(rows).sort_values(['band','detector'])
print(df.to_string(index=False))

print('\nBREAKDOWN CRITERION (Eq. 9 in the paper)')
for model, bd in res['breakdown'].items():
    bad = [b for b, v in bd.items() if v['breaks_down']]
    print(f'  {model:16} breaks down in: {", ".join(bad) if bad else "no band"}')

print('\nDOES DETECTION DEGRADE WITH INTERVAL?')
for model, bands in res['per_band'].items():
    drs = [bands[b]['DR'] for b in ('B1','B2','B3') if b in bands]
    if len(drs) == 3:
        mono = drs[0] >= drs[1] >= drs[2]
        print(f'  {model:16} {drs[0]:.3f} -> {drs[1]:.3f} -> {drs[2]:.3f}   '
              f'{"monotonic decline (supports the paper)" if mono else "NOT monotonic — REWRITE Section V"}')

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for f in ['fig_detection_far','fig_roc','fig_confusion','fig_interval_hist','fig_importance']:
    p = Path('figures')/f'{f}.png'
    if p.exists():
        print(f'\n=== {f} ==='); display(Image(str(p)))

---
## Step 8 — Export everything you need

Downloads a zip containing the figures, the generated LaTeX table and macros, the
results JSON, and `derived_channel_intervals.csv` — the derived interval labels, which
are the artefact your paper promises to release.

In [ ]:
import shutil, os, json
from pathlib import Path

out = Path('/content/quiet_signal_outputs'); shutil.rmtree(out, ignore_errors=True)
(out/'figures').mkdir(parents=True); (out/'paper').mkdir(); (out/'results').mkdir()

for p in Path('figures').glob('*'):   shutil.copy(p, out/'figures'/p.name)
for p in Path('results').glob('*'):   shutil.copy(p, out/'results'/p.name)
for n in ('results_table.tex','results_macros.tex','dose_response_macros.tex'):
    if Path('paper',n).exists(): shutil.copy(Path('paper',n), out/'paper'/n)

shutil.make_archive('/content/quiet_signal_outputs', 'zip', out)
sz = os.path.getsize('/content/quiet_signal_outputs.zip')/1e6
print(f'quiet_signal_outputs.zip  ({sz:.1f} MB)')

try:
    from google.colab import files
    files.download('/content/quiet_signal_outputs.zip')
except Exception as e:
    print('Download it from the file pane on the left. (', e, ')')

### Optional — save to Google Drive so a session timeout cannot cost you the run

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/quiet_signal_outputs.zip', '/content/drive/MyDrive/')
# print('saved to Drive')

---
## Step 9 — Finish the paper

You now have real numbers. Six things stand between you and a submittable manuscript.

**1. Drop the results into the LaTeX project.**
Unzip the export and copy `paper/results_macros.tex` and `paper/results_table.tex` over
the placeholder versions in your Overleaf project, and put the PDFs from `figures/`
into `paper/figures/`. The manuscript reads every number from those macros — there is no
way for the prose and your data to disagree.

**2. Add your Fig. 1.** Export your Mermaid architecture diagram as SVG, convert to PDF,
and save it as `paper/figures/fig_architecture.pdf`, replacing the placeholder.

**3. Turn off draft mode.** In `main.tex` set `\newcommand{\DraftResults}{0}`. The red
banner disappears. In the Word version, delete the banner paragraph. **Do not submit
while either is visible.**

**4. Check the CTU-13 claim against what you ran.** If Step 3b downloaded scenarios and
they appear in the Step 4 census, the paper's cross-corpus claim is supported as written
— nothing to change. If Step 3b came back empty and you proceeded on IoT-23 alone, then
the claim is not supported and you must edit these places:
- *Abstract*: "two openly released corpora" → "an openly released corpus"
- *Section III-C*: remove the CTU-13 column from Table I and the cross-corpus sentences
- *Section IV-B*: remove "Cross-corpus validation is additionally performed…"
- *Section VI, Limitations*: state plainly that validation is single-corpus

**5. Check Section V against reality.** If Step 7 said *NOT monotonic*, your results
contradict the paper's central claim. Rewrite Section V and the abstract to describe
what you found. This is the most important instruction in this notebook.

**6. Push the code and fix the citation.** Upload `/content/quiet-signal` to GitHub,
then replace `USERNAME` in `references.bib`, `CITATION.cff` and `README.md`. Reference
[19] in the paper is your repository — a broken link there is an easy reviewer complaint.

**7. If you ran Step 6b, write up the extension honestly.** Both the dose-response
crossing points and the cross-corpus numbers are exploratory — they were not fixed
before this run's results existed, so the pre-registration claim in Section III-I does
not cover them. Say that explicitly wherever you report them (a short subsection or an
appendix, not folded into the main results as if pre-registered). If cross-corpus came
back empty because only one corpus was under `data/`, do not claim cross-corpus
generalisation was tested — say it was not.

---

### Final checklist before you submit

- [ ] `\DraftResults` is `0`; no `[TBD]` remains — search the PDF for it
- [ ] Fig. 1 is your diagram, not the placeholder
- [ ] Every figure is watermark-free (re-exported from this run, not the samples)
- [ ] Page count ≤ 8 in the real IEEEtran build on Overleaf
- [ ] GitHub link resolves
- [ ] Claims about CTU-13 match what you actually ran (Step 4 census is the evidence)
- [ ] Section V describes your data, not the hypothesis
- [ ] If Step 6b results are reported, they are labelled exploratory, not pre-registered